# **ESG DART 기말 프로젝트**

## **Phase 1: 데이터 수집**

한국 상장기업 사업보고서(DART)에서 ESG 관련 텍스트를 추출하고
KCGS ESG 등급과의 연관성을 분석한다.

### **1-1. 환경 설정 및 OpenDART API 연결 확인**

In [4]:
import os
import json
import time
import zipfile
from io import BytesIO

import requests
import pandas as pd
from dotenv import load_dotenv

# .env 로드
load_dotenv()
API_KEY = os.getenv("OPENDART_API_KEY") or os.getenv("DART_API_KEY")

# API 키 존재 확인 (값 자체는 출력 금지 — 제출 시 노출 방지)
assert API_KEY is not None, ".env에서 API 키를 찾을 수 없음"
print(f"API 키 로드 성공 (길이: {len(API_KEY)}자, 앞 4자: {API_KEY[:4]}***)")

# 경로 설정
DATA_DIR = "data"
OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"DATA_DIR: {DATA_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

API 키 로드 성공 (길이: 40자, 앞 4자: 0081***)
DATA_DIR: data
OUTPUT_DIR: outputs


In [5]:
# OpenDART API 연결 테스트 (삼성전자 corp_code로 간단 조회)
# 공시검색 API: https://opendart.fss.or.kr/api/list.json

test_params = {
    "crtfc_key": API_KEY,
    "corp_code": "00126380",   # 삼성전자 (알려진 값으로 연결만 테스트)
    "bgn_de": "20250101",
    "end_de": "20251231",
    "pblntf_detail_ty": "A001",  # 사업보고서
    "page_count": "10",
}

res = requests.get(
    "https://opendart.fss.or.kr/api/list.json",
    params=test_params,
    timeout=20,
)

print(f"HTTP 상태: {res.status_code}")

payload = res.json()
print(f"API 응답 status: {payload.get('status')}")
print(f"API 응답 message: {payload.get('message')}")

# status 코드 의미:
#   '000' = 정상
#   '010' = 등록되지 않은 키
#   '011' = 사용할 수 없는 키 (오늘 한도 초과 등)
#   '013' = 조회 데이터 없음
#   '020' = 요청 제한 초과

HTTP 상태: 200
API 응답 status: 000
API 응답 message: 정상


### **1-2. company_master.csv 구조 확인**

분석의 출발점인 `company_master.csv`를 불러와 구조를 파악한다.

**핵심 식별자 3종**
- **`stock_code`**: 종목코드 (CSV에 이미 존재). KCGS 등급 연결 키
- **`corp_code`**: OpenDART 내부 회사 코드 (비어 있음 → 직접 채워야 함)
- **`rcept_no`**: 사업보고서 접수번호 (비어 있음 → 직접 채워야 함)

**분석 단위 주의:** 표본 단위는 회사명이 아니라 **`stock_code × fiscal_year`**.
같은 종목코드가 사명변경·분할로 여러 이름을 가질 수 있어 회사명 join은 위험.

**타이밍 규칙:** `esg_year = fiscal_year + 1`
(2024 사업보고서 ↔ 2025 ESG 등급)

In [6]:
# company_master.csv 로드
cm = pd.read_csv(os.path.join(DATA_DIR, "company_master.csv"))

print(f"shape: {cm.shape}  (행 {cm.shape[0]}, 열 {cm.shape[1]})")
print(f"\n컬럼 목록:")
for col in cm.columns:
    print(f"  - {col}")

shape: (381, 19)  (행 381, 열 19)

컬럼 목록:
  - team_id
  - role
  - company_name
  - corp_code
  - stock_code
  - industry
  - fiscal_year
  - report_code
  - rcept_no
  - esg_source
  - esg_grade
  - e_grade
  - s_grade
  - g_grade
  - esg_year
  - is_required
  - replacement_of
  - replacement_reason
  - notes


In [7]:
# 핵심 식별자 컬럼 상태 확인
print("[식별자 컬럼 상태]\n")

# stock_code: 채워져 있어야 함
print(f"stock_code 결측: {cm['stock_code'].isna().sum()} / {len(cm)}")
print(f"stock_code 고유값: {cm['stock_code'].nunique()}개")

# corp_code: 비어 있어야 정상 (직접 채울 예정)
print(f"\ncorp_code 결측: {cm['corp_code'].isna().sum()} / {len(cm)}")

# rcept_no: 비어 있어야 정상
print(f"rcept_no 결측: {cm['rcept_no'].isna().sum()} / {len(cm)}")

# 분석 단위 점검 — 회사명 vs 종목코드 불일치 확인
print(f"\n[분석 단위 점검]")
print(f"company_name 고유: {cm['company_name'].nunique()}개")
print(f"stock_code 고유:   {cm['stock_code'].nunique()}개")
print(f"→ 회사명으로 join하면 silent 오매칭 위험 (수가 다름)")

# 연도 구조 확인
print(f"\n[연도 구조]")
print(f"fiscal_year 분포:\n{cm['fiscal_year'].value_counts().sort_index()}")
print(f"\nesg_year 분포:\n{cm['esg_year'].value_counts().sort_index()}")

[식별자 컬럼 상태]

stock_code 결측: 0 / 381
stock_code 고유값: 127개

corp_code 결측: 381 / 381
rcept_no 결측: 381 / 381

[분석 단위 점검]
company_name 고유: 133개
stock_code 고유:   127개
→ 회사명으로 join하면 silent 오매칭 위험 (수가 다름)

[연도 구조]
fiscal_year 분포:
fiscal_year
2022    127
2023    127
2024    127
Name: count, dtype: int64

esg_year 분포:
esg_year
2022    127
2023    127
2024    127
Name: count, dtype: int64


In [8]:
# ESG 등급 컬럼 확인 (KCGS 등급)
print("[ESG 등급 분포]\n")

for col in ['esg_grade', 'e_grade', 's_grade', 'g_grade']:
    if col in cm.columns:
        print(f"{col}:")
        print(cm[col].value_counts().sort_index().to_dict())
        print(f"  결측: {cm[col].isna().sum()}\n")

# 처음 3행 미리보기 (식별자 + 등급만)
print("[처음 3행 — 주요 컬럼]")
preview_cols = ['company_name', 'stock_code', 'corp_code', 'fiscal_year', 
                'esg_year', 'esg_grade', 'industry']
preview_cols = [c for c in preview_cols if c in cm.columns]
print(cm[preview_cols].head(3).to_string(index=False))

[ESG 등급 분포]

esg_grade:
{'A': 115, 'A+': 23, 'B': 38, 'B+': 77, 'C': 57, 'D': 71}
  결측: 0

e_grade:
{'A': 113, 'A+': 30, 'B': 32, 'B+': 71, 'C': 78, 'D': 57}
  결측: 0

s_grade:
{'A': 95, 'A+': 112, 'B': 19, 'B+': 48, 'C': 34, 'D': 73}
  결측: 0

g_grade:
{'A': 76, 'A+': 18, 'B': 57, 'B+': 104, 'C': 60, 'D': 66}
  결측: 0

[처음 3행 — 주요 컬럼]
company_name  stock_code  corp_code  fiscal_year  esg_year esg_grade industry
        삼성전자        5930        NaN         2022      2022         A     전기전자
        삼성전자        5930        NaN         2023      2023         A     전기전자
        삼성전자        5930        NaN         2024      2024        B+     전기전자


In [ ]:
# 확인 사항

# 행 수 - 381
# corp_code, rcept_no 결측 여부
# company_name 133 vs stock_code 127 불일치 확인
# ESG 등급 컬럼들 생김새 (등급 값 형식)

### **1-3. stock_code 정규화**

`stock_code`가 CSV 로드 시 정수로 인식되어 앞자리 0이 손실
(`005930` → `5930`) 6자리 문자열로 복구.

**`esg_year` 주의:** CSV의 `esg_year`는 `fiscal_year`와 동일하게 표기되어 있다.  
가이드의 타이밍 규칙(`esg_year = fiscal_year + 1`)은 **사업보고서 검색 시
`fiscal_year + 1`년 공시를 찾는 것**으로 적용한다.  
이 처리는 식별자 매핑단계(1-4)에서 명시적으로 수행한다.

In [10]:
# stock_code를 6자리 문자열로 정규화 (앞자리 0 복구)
cm['stock_code'] = cm['stock_code'].astype(str).str.zfill(6)

print("[stock_code 정규화 후]")
print(cm[['company_name', 'stock_code', 'fiscal_year', 'esg_year']].head(6).to_string(index=False))

# 검증: 모든 stock_code가 6자리인지
len_check = cm['stock_code'].str.len().value_counts()
print(f"\nstock_code 길이 분포: {len_check.to_dict()}")
assert (cm['stock_code'].str.len() == 6).all(), "6자리 아닌 종목코드 존재"
print("→ 모든 stock_code가 6자리 ✓")

# 고유 종목코드 확인
unique_stocks = cm['stock_code'].unique()
print(f"\n고유 종목코드: {len(unique_stocks)}개")
print(f"예시 5개: {sorted(unique_stocks)[:5]}")

[stock_code 정규화 후]
company_name stock_code  fiscal_year  esg_year
        삼성전자     005930         2022      2022
        삼성전자     005930         2023      2023
        삼성전자     005930         2024      2024
         BYC     001460         2022      2022
         BYC     001460         2023      2023
         BYC     001460         2024      2024

stock_code 길이 분포: {6: 381}
→ 모든 stock_code가 6자리 ✓

고유 종목코드: 127개
예시 5개: ['000020', '000040', '000050', '000070', '000080']


### **1-4. corp_code 매핑 — corpCode.xml 전체 다운로드**

OpenDART는 전체 등록 기업의 `corp_code ↔ stock_code` 매핑을
`corpCode.xml`(ZIP) 하나로 제공한다. 개별 기업명 검색(127회 호출 +
오매칭 위험) 대신 이 파일 1개로 전체를 매핑한다.

**재현성:** 같은 파일을 받으면 항상 같은 매핑. API 호출 1회.

**매핑 키:** 회사명이 아니라 `stock_code`로 join (silent 오매칭 방지)

In [11]:
# OpenDART 전체 기업 코드 파일 다운로드 (ZIP 안에 CORPCODE.xml)
corpcode_zip_path = os.path.join(OUTPUT_DIR, "corpCode.zip")

if os.path.exists(corpcode_zip_path):
    print("[corpCode.zip 캐시 존재 — 다운로드 스킵]")
else:
    print("[corpCode.xml 다운로드 중...]")
    res = requests.get(
        "https://opendart.fss.or.kr/api/corpCode.xml",
        params={"crtfc_key": API_KEY},
        timeout=30,
    )
    res.raise_for_status()
    
    # 응답이 ZIP인지 확인 (에러 시 JSON/XML 에러 메시지가 올 수 있음)
    content_type = res.headers.get("Content-Type", "")
    print(f"  Content-Type: {content_type}")
    print(f"  응답 크기: {len(res.content):,} bytes")
    
    # ZIP 저장
    with open(corpcode_zip_path, "wb") as f:
        f.write(res.content)
    print(f"  저장: {corpcode_zip_path}")

[corpCode.zip 캐시 존재 — 다운로드 스킵]


In [12]:
import xml.etree.ElementTree as ET

# ZIP 안의 CORPCODE.xml 파싱
with zipfile.ZipFile(corpcode_zip_path) as zf:
    xml_name = zf.namelist()[0]
    print(f"ZIP 내부 파일: {xml_name}")
    xml_bytes = zf.read(xml_name)

root = ET.fromstring(xml_bytes)

# 각 <list> 항목: corp_code, corp_name, stock_code, modify_date
records = []
for item in root.iter("list"):
    records.append({
        "corp_code":  item.findtext("corp_code", "").strip(),
        "corp_name":  item.findtext("corp_name", "").strip(),
        "stock_code": item.findtext("stock_code", "").strip(),
        "modify_date": item.findtext("modify_date", "").strip(),
    })

corp_df = pd.DataFrame(records)
print(f"\n전체 등록 기업: {len(corp_df):,}개")

# 상장사만 (stock_code가 있는 것) — 비상장은 stock_code가 빈 문자열
listed = corp_df[corp_df["stock_code"].str.strip() != ""].copy()
listed["stock_code"] = listed["stock_code"].str.zfill(6)
print(f"상장사 (stock_code 보유): {len(listed):,}개")

print(f"\n[예시 3개]")
print(listed.head(3).to_string(index=False))

ZIP 내부 파일: CORPCODE.xml

전체 등록 기업: 118,083개
상장사 (stock_code 보유): 3,965개

[예시 3개]
corp_code corp_name stock_code modify_date
 00260985      한빛네트     036720    20170630
 00264529      엔플렉스     040130    20170630
 00358545    동서정보기술     055000    20170630


In [13]:
# company_master의 stock_code에 corp_code 매핑
stock_to_corp = dict(zip(listed["stock_code"], listed["corp_code"]))

cm["corp_code"] = cm["stock_code"].map(stock_to_corp)

# 매핑 결과 점검
n_total = len(cm)
n_mapped = cm["corp_code"].notna().sum()
n_missing = cm["corp_code"].isna().sum()

print(f"[corp_code 매핑 결과]")
print(f"  전체 행: {n_total}")
print(f"  매핑 성공: {n_mapped}")
print(f"  매핑 실패: {n_missing}")

# 매핑 실패한 종목코드 확인 (있다면)
if n_missing > 0:
    missing_stocks = cm[cm["corp_code"].isna()][
        ["company_name", "stock_code", "fiscal_year"]
    ].drop_duplicates(subset=["stock_code"])
    print(f"\n[매핑 실패 종목코드 — {missing_stocks['stock_code'].nunique()}개]")
    print(missing_stocks.to_string(index=False))
else:
    print(f"\n→ 127개 전체 매핑 성공 ✓")

# 삼성전자 확인 (sanity check)
samsung = cm[cm["stock_code"] == "005930"][
    ["company_name", "stock_code", "corp_code", "fiscal_year", "esg_year"]
]
print(f"\n[삼성전자 매핑 확인]")
print(samsung.to_string(index=False))
print(f"\n(삼성전자 corp_code는 00126380이 정상)")

[corp_code 매핑 결과]
  전체 행: 381
  매핑 성공: 381
  매핑 실패: 0

→ 127개 전체 매핑 성공 ✓

[삼성전자 매핑 확인]
company_name stock_code corp_code  fiscal_year  esg_year
        삼성전자     005930  00126380         2022      2022
        삼성전자     005930  00126380         2023      2023
        삼성전자     005930  00126380         2024      2024

(삼성전자 corp_code는 00126380이 정상)


### **1-5. 사업보고서 rcept_no 찾기 (삼성전자 연습)**

가이드 권장에 따라 먼저 1개 기업-연도로 절차를 검증한다.

**타이밍 처리:** `fiscal_year=2024`의 사업보고서는 `fiscal_year+1=2025`년에
공시된다. 따라서:
- DART 공시검색 기간: `bgn_de=20250101`, `end_de=20251231`
- 보고서명 필터: `"사업보고서"` 포함 + `"(2024.12)"` 포함
- 정정공시가 있으면 어떤 rcept_no를 썼는지 기록

**선택 기준은 제출 월이 아니라 보고서명과 대상 회계기간**이다.

In [14]:
def find_business_report(corp_code, fiscal_year, api_key, verbose=True):
    """
    특정 corp_code의 특정 회계연도(fiscal_year) 사업보고서를 찾는다.
    fiscal_year의 사업보고서는 fiscal_year+1년에 공시되므로
    검색 기간을 fiscal_year+1년으로 설정.
    
    Returns: dict (rcept_no, report_nm, rcept_dt, ...) 또는 None
    """
    search_year = fiscal_year + 1   # 타이밍 규칙
    
    params = {
        "crtfc_key": api_key,
        "corp_code": corp_code,
        "bgn_de": f"{search_year}0101",
        "end_de": f"{search_year}1231",
        "pblntf_detail_ty": "A001",   # 사업보고서
        "page_count": "100",
    }
    
    res = requests.get(
        "https://opendart.fss.or.kr/api/list.json",
        params=params,
        timeout=20,
    )
    payload = res.json()
    
    status = payload.get("status")
    if status != "000":
        if verbose:
            print(f"  API status={status}: {payload.get('message')}")
        return None
    
    rows = payload.get("list", [])
    if verbose:
        print(f"  검색 기간 {search_year}년: {len(rows)}개 공시 발견")
    
    # 보고서명에 "사업보고서" + 대상 회계연도 "(fiscal_year.12)" 포함된 것
    target_tag = f"({fiscal_year}.12)"
    candidates = [
        r for r in rows
        if "사업보고서" in r.get("report_nm", "")
        and target_tag in r.get("report_nm", "")
    ]
    
    if verbose:
        print(f"  '사업보고서' + '{target_tag}' 필터 후: {len(candidates)}개")
        for c in candidates:
            print(f"    - {c.get('report_nm')} | rcept_no={c.get('rcept_no')} | {c.get('rcept_dt')}")
    
    if not candidates:
        return None
    
    # 여러 개면 가장 최근 접수 (정정공시가 원본보다 나중)
    chosen = sorted(candidates, key=lambda r: r.get("rcept_dt", ""), reverse=True)[0]
    
    return {
        "corp_code": corp_code,
        "fiscal_year": fiscal_year,
        "report_nm": chosen.get("report_nm"),
        "rcept_no": chosen.get("rcept_no"),
        "rcept_dt": chosen.get("rcept_dt"),
        "n_candidates": len(candidates),
        "all_report_nms": [c.get("report_nm") for c in candidates],
    }


# 삼성전자 2024 회계연도 사업보고서 찾기
print("[삼성전자 fiscal_year=2024 사업보고서 검색]")
result = find_business_report("00126380", 2024, API_KEY, verbose=True)

print(f"\n[선택 결과]")
if result:
    for k, v in result.items():
        print(f"  {k}: {v}")
    viewer_url = f"https://dart.fss.or.kr/dsaf001/main.do?rcpNo={result['rcept_no']}"
    print(f"  viewer_url: {viewer_url}")
else:
    print("  사업보고서를 찾지 못함")

[삼성전자 fiscal_year=2024 사업보고서 검색]
  검색 기간 2025년: 1개 공시 발견
  '사업보고서' + '(2024.12)' 필터 후: 1개
    - 사업보고서 (2024.12) | rcept_no=20250311001085 | 20250311

[선택 결과]
  corp_code: 00126380
  fiscal_year: 2024
  report_nm: 사업보고서 (2024.12)
  rcept_no: 20250311001085
  rcept_dt: 20250311
  n_candidates: 1
  all_report_nms: ['사업보고서 (2024.12)']
  viewer_url: https://dart.fss.or.kr/dsaf001/main.do?rcpNo=20250311001085


### **1-6. 사업보고서 원문 다운로드 (document.xml)**

`rcept_no`로 OpenDART `document.xml` API를 호출하면 보고서 원문이
ZIP으로 온다. ZIP 안에는 XML 파일이 있고, 여기서 텍스트를 추출한다.

**주의 (가이드):** MCP가 생성한 passage는 실제 공시와 다를 수 있으므로,
반드시 원문 XML을 직접 파싱해 재현한다. 이 단계가 그 재현이다.

In [15]:
def download_document_xml(rcept_no, api_key, output_dir, verbose=True):
    """
    rcept_no의 사업보고서 원문 ZIP을 다운로드하고 본 보고서 XML 텍스트를 반환.
    
    ZIP 안에는 여러 XML이 있을 수 있음
      - {rcept_no}.xml 
      - {rcept_no}_NNNNN.xml 
    본 보고서만 선택.
    """
    zip_path = os.path.join(output_dir, f"doc_{rcept_no}.zip")
    
    if os.path.exists(zip_path):
        if verbose:
            print(f"  캐시 존재: {zip_path}")
    else:
        if verbose:
            print(f"  다운로드 중: rcept_no={rcept_no}")
        res = requests.get(
            "https://opendart.fss.or.kr/api/document.xml",
            params={"crtfc_key": api_key, "rcept_no": rcept_no},
            timeout=60,
        )
        res.raise_for_status()
        content_type = res.headers.get("Content-Type", "")
        if verbose:
            print(f"  Content-Type: {content_type}")
            print(f"  응답 크기: {len(res.content):,} bytes")
        with open(zip_path, "wb") as f:
            f.write(res.content)
        if verbose:
            print(f"  저장: {zip_path}")
    
    # ZIP 내부 XML 목록
    with zipfile.ZipFile(zip_path) as zf:
        names = zf.namelist()
        if verbose:
            print(f"  ZIP 내부 파일: {names}")
        
        # 본 보고서 XML 선택: 이름이 정확히 "{rcept_no}.xml"
        main_name = f"{rcept_no}.xml"
        if main_name in names:
            chosen_name = main_name
        else:
            # fallback: suffix 없는(언더스코어 없는) xml 중 첫 번째
            no_suffix = [n for n in names if "_" not in n and n.endswith(".xml")]
            chosen_name = no_suffix[0] if no_suffix else names[0]
        
        if verbose:
            print(f"  → 본 보고서로 선택: {chosen_name}")
        xml_bytes = zf.read(chosen_name)
    
    # 디코딩
    for enc in ("utf-8", "euc-kr", "cp949"):
        try:
            xml_text = xml_bytes.decode(enc)
            if verbose:
                print(f"  디코딩 성공: {enc}")
            break
        except UnicodeDecodeError:
            continue
    else:
        xml_text = xml_bytes.decode("utf-8", errors="ignore")
        if verbose:
            print(f"  디코딩: utf-8 (errors=ignore)")
    
    return xml_text, zip_path, chosen_name


# 삼성전자 원문 다시 다운로드 (캐시 ZIP은 그대로, XML 선택만 수정)
print("[삼성전자 사업보고서 원문 — 본 보고서 XML 선택]")
xml_text, zip_path, chosen_name = download_document_xml(
    "20250311001085", API_KEY, OUTPUT_DIR, verbose=True
)

print(f"\n선택된 XML: {chosen_name}")
print(f"원문 XML 길이: {len(xml_text):,}자")
print(f"\n[처음 800자 미리보기]")
print(xml_text[:800])

[삼성전자 사업보고서 원문 — 본 보고서 XML 선택]
  캐시 존재: outputs\doc_20250311001085.zip
  ZIP 내부 파일: ['20250311001085_00761.xml', '20250311001085_00760.xml', '20250311001085.xml']
  → 본 보고서로 선택: 20250311001085.xml
  디코딩 성공: utf-8

선택된 XML: 20250311001085.xml
원문 XML 길이: 5,780,877자

[처음 800자 미리보기]
<?xml version="1.0" encoding="utf-8"?>


<DOCUMENT xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:noNamespaceSchemaLocation="dart4.xsd">

<DOCUMENT-NAME ACODE="11011">사업보고서</DOCUMENT-NAME>
<FORMULA-VERSION ADATE="20241231">6.0</FORMULA-VERSION>
<COMPANY-NAME AREGCIK="00126380">삼성전자주식회사</COMPANY-NAME>

<SUMMARY>
<EXTRACTION ACODE="LINK_FLAG" AFEATURE="BOTH">C</EXTRACTION>
<EXTRACTION ACODE="FIN_TYPE" AFEATURE="BOTH">A</EXTRACTION>
<EXTRACTION ACODE="IFRS_YN" AFEATURE="BOTH">Y</EXTRACTION>
<EXTRACTION ACODE="CRP_RGS_NO_TEMP" AFEATURE="BOTH">130111-0006246</EXTRACTION>
</SUMMARY>


<BODY ATOCID="558">

<COVER>
<P></P>
<COVER-TITLE ATOC="Y" AASSOCNOTE="COVER" ATOCID="402" ENG="Annual Report">사 업 보 고 서</C

### **1-7. 섹션 구조 진단**

DART 사업보고서는 `II. 사업의 내용`, `IV. 이사의 경영진단`,
`VI. 이사회 등 회사의 기관` 등으로 구성된다. 추출 로직을 짜기 전에
이 섹션들이 XML에서 어떤 태그/속성으로 표현되는지 확인한다.

In [16]:
import re

# DART 보고서의 대분류 섹션은 보통 <TITLE> 또는 <LIBRARY>/<SECTION> 태그에
# "I.", "II.", ... 로마숫자 + 제목 형태로 등장

# 1) TITLE 태그 내용 모두 추출
titles = re.findall(r'<TITLE[^>]*>(.*?)</TITLE>', xml_text, re.DOTALL)
print(f"[TITLE 태그 총 {len(titles)}개]")
print("처음 30개:")
for i, t in enumerate(titles[:30]):
    clean = re.sub(r'\s+', ' ', t).strip()
    if clean:
        print(f"  {i:3}: {clean[:80]}")

[TITLE 태그 총 135개]
처음 30개:
    0: 목 차
    1: 【 대표이사 등의 확인 】
    2: I. 회사의 개요
    3: 1. 회사의 개요
    4: 2. 회사의 연혁
    5: 3. 자본금 변동사항
    6: 4. 주식의 총수 등
    7: 5. 정관에 관한 사항
    8: II. 사업의 내용
    9: 1. 사업의 개요
   10: 2. 주요 제품 및 서비스
   11: 3. 원재료 및 생산설비
   12: 4. 매출 및 수주상황
   13: 5. 위험관리 및 파생거래
   14: 6. 주요계약 및 연구개발활동
   15: 7. 기타 참고사항
   16: III. 재무에 관한 사항
   17: 1. 요약재무정보
   18: 2. 연결재무제표
   19: 2-1. 연결 재무상태표
   20: 2-2. 연결 손익계산서
   21: 2-3. 연결 포괄손익계산서
   22: 2-4. 연결 자본변동표
   23: 2-5. 연결 현금흐름표
   24: 3. 연결재무제표 주석
   25: 1. 일반적 사항 (연결)
   26: 2. 중요한 회계처리방침 (연결)
   27: 3. 중요한 회계추정 및 가정 (연결)
   28: 4. 범주별 금융상품 (연결)
   29: 5. 금융자산의 양도 (연결)


In [17]:
# 로마숫자로 시작하는 대분류 제목 패턴 탐색
# 예: "II. 사업의 내용", "IV. 이사의 경영진단 및 분석의견"

roman_pattern = re.compile(
    r'(I{1,3}|IV|V|VI{0,3}|IX|X)\.\s*([가-힣].{2,40})'
)

# TITLE 태그 안에서 로마숫자 섹션 찾기
print("[로마숫자 대분류 섹션 후보]")
section_hits = []
for t in titles:
    clean = re.sub(r'\s+', ' ', t).strip()
    m = roman_pattern.match(clean)
    if m:
        section_hits.append(clean)
        print(f"  {clean[:70]}")

print(f"\n총 {len(section_hits)}개 로마숫자 섹션 발견")

# 우리가 원하는 3개 섹션이 있는지 확인
targets = {
    "II": "사업의 내용",
    "IV": "이사의 경영진단",
    "VI": "이사회",
}
print(f"\n[목표 섹션 존재 여부]")
for roman, keyword in targets.items():
    found = [s for s in section_hits if keyword in s]
    status = "✓" if found else "✗"
    print(f"  {status} {roman}. {keyword}: {found[:1]}")

[로마숫자 대분류 섹션 후보]
  I. 회사의 개요
  II. 사업의 내용
  III. 재무에 관한 사항
  IV. 이사의 경영진단 및 분석의견
  V. 회계감사인의 감사의견 등
  VI. 이사회 등 회사의 기관에 관한 사항
  VII. 주주에 관한 사항
  VIII. 임원 및 직원 등에 관한 사항
  IX. 계열회사 등에 관한 사항
  X. 대주주 등과의 거래내용

총 10개 로마숫자 섹션 발견

[목표 섹션 존재 여부]
  ✓ II. 사업의 내용: ['II. 사업의 내용']
  ✓ IV. 이사의 경영진단: ['IV. 이사의 경영진단 및 분석의견']
  ✓ VI. 이사회: ['VI. 이사회 등 회사의 기관에 관한 사항']


In [18]:
# XML에 어떤 태그들이 쓰이는지 빈도 확인 (구조 파악)
all_tags = re.findall(r'<([A-Z][A-Z0-9-]*)', xml_text)
from collections import Counter
tag_counts = Counter(all_tags)

print("[XML 태그 빈도 — 상위 20개]")
for tag, cnt in tag_counts.most_common(20):
    print(f"  {tag}: {cnt:,}")

# 섹션 구분에 쓰일 만한 태그 확인
print(f"\n[구조 태그 존재 여부]")
for tag in ['TITLE', 'SECTION-1', 'SECTION-2', 'LIBRARY', 'PART', 'P', 'TABLE', 'TU', 'TE']:
    print(f"  {tag}: {tag_counts.get(tag, 0):,}개")

[XML 태그 빈도 — 상위 20개]
  TD: 27,490
  TE: 16,149
  P: 15,627
  TR: 10,417
  COL: 5,673
  TH: 5,128
  TABLE: 2,027
  COLGROUP: 2,027
  TBODY: 2,027
  SPAN: 898
  THEAD: 613
  TU: 591
  PGBRK: 204
  TABLE-GROUP: 138
  TITLE: 135
  SECTION-2: 37
  LIBRARY: 25
  SECTION-1: 14
  A: 8
  EXTRACTION: 4

[구조 태그 존재 여부]
  TITLE: 135개
  SECTION-1: 14개
  SECTION-2: 37개
  LIBRARY: 25개
  PART: 0개
  P: 15,627개
  TABLE: 2,027개
  TU: 591개
  TE: 16,149개


### **1-8. II / IV / VI 섹션 텍스트 추출**

진단 결과 섹션 제목은 `<TITLE>` 태그에 로마숫자 형식으로 존재한다.
목표 3개 섹션을 추출한다.

**추출 전략:**
- 섹션 경계: 대상 섹션 `<TITLE>` ~ 다음 대분류 `<TITLE>` 직전
- 본문: `<P>` 태그 텍스트만 (표 `<TABLE>`은 재무 숫자라 제외)
- 추출 후 섹션별 출처 기록 (가이드 요건)

**선택한 3개 섹션 (가이드 지정):**
- **II. 사업의 내용** — 사업 개요, ESG 활동 서술
- **IV. 이사의 경영진단 및 분석의견** — 리스크, 경영 평가
- **VI. 이사회 등 회사의 기관에 관한 사항** — 지배구조

In [19]:
import html

def extract_esg_sections(xml_text, verbose=True):
    """
    DART 사업보고서 XML에서 II, IV, VI 섹션의 <P> 텍스트를 추출.
    
    Returns: dict {
        'II': {'title': ..., 'text': ..., 'n_paragraphs': ...},
        'IV': {...},
        'VI': {...},
    }
    """
    # 1) 모든 TITLE의 위치와 내용 찾기
    title_iter = list(re.finditer(r'<TITLE[^>]*>(.*?)</TITLE>', xml_text, re.DOTALL))
    
    # 2) 로마숫자 대분류 TITLE만 골라 (위치, 로마숫자, 제목) 기록
    roman_re = re.compile(r'^\s*(I{1,3}|IV|V|VI{0,3}|IX|X)\.\s*([가-힣].{1,40})')
    big_sections = []  # (start_pos, roman, title_text)
    for m in title_iter:
        clean = re.sub(r'\s+', ' ', m.group(1)).strip()
        rm = roman_re.match(clean)
        if rm:
            big_sections.append({
                "pos": m.start(),
                "roman": rm.group(1),
                "title": clean,
            })
    
    if verbose:
        print(f"[대분류 섹션 {len(big_sections)}개 경계 확인]")
        for s in big_sections:
            print(f"  {s['roman']:>4} | pos={s['pos']:>9,} | {s['title']}")
    
    # 3) 목표 섹션의 [시작, 끝] 범위 계산
    targets = ["II", "IV", "VI"]
    result = {}
    
    for i, sec in enumerate(big_sections):
        if sec["roman"] not in targets:
            continue
        start = sec["pos"]
        # 끝 = 다음 대분류 섹션의 시작 (없으면 문서 끝)
        end = big_sections[i + 1]["pos"] if i + 1 < len(big_sections) else len(xml_text)
        
        section_xml = xml_text[start:end]
        
        # <P> 태그 텍스트만 추출 (TABLE 내부 P는 제외하기 위해 TABLE 블록 먼저 제거)
        section_no_table = re.sub(r'<TABLE.*?</TABLE>', ' ', section_xml, flags=re.DOTALL)
        
        # <P> 내용 추출
        paragraphs = re.findall(r'<P[^>]*>(.*?)</P>', section_no_table, re.DOTALL)
        
        clean_paragraphs = []
        for p in paragraphs:
            # 내부 태그 제거
            txt = re.sub(r'<[^>]+>', ' ', p)
            # HTML 엔티티 복원 (&amp; &lt; 등)
            txt = html.unescape(txt)
            # 공백 정규화
            txt = re.sub(r'\s+', ' ', txt).strip()
            if len(txt) >= 10:   # 너무 짧은 건 제외 (빈 P, 기호만 등)
                clean_paragraphs.append(txt)
        
        full_text = "\n".join(clean_paragraphs)
        
        result[sec["roman"]] = {
            "title": sec["title"],
            "text": full_text,
            "n_paragraphs": len(clean_paragraphs),
            "n_chars": len(full_text),
        }
    
    return result


# 삼성전자에 적용
print("[삼성전자 II/IV/VI 섹션 추출]\n")
sections = extract_esg_sections(xml_text, verbose=True)

print(f"\n[추출 결과 요약]")
for roman in ["II", "IV", "VI"]:
    if roman in sections:
        s = sections[roman]
        print(f"\n  [{roman}] {s['title']}")
        print(f"    문단 수: {s['n_paragraphs']:,}")
        print(f"    글자 수: {s['n_chars']:,}")
        print(f"    앞 200자: {s['text'][:200]}")

[삼성전자 II/IV/VI 섹션 추출]

[대분류 섹션 10개 경계 확인]
     I | pos=    5,451 | I. 회사의 개요
    II | pos=  138,075 | II. 사업의 내용
   III | pos=  282,560 | III. 재무에 관한 사항
    IV | pos=3,235,993 | IV. 이사의 경영진단 및 분석의견
     V | pos=3,294,473 | V. 회계감사인의 감사의견 등
    VI | pos=3,393,935 | VI. 이사회 등 회사의 기관에 관한 사항
   VII | pos=3,558,575 | VII. 주주에 관한 사항
  VIII | pos=3,637,963 | VIII. 임원 및 직원 등에 관한 사항
    IX | pos=4,590,122 | IX. 계열회사 등에 관한 사항
     X | pos=4,903,307 | X. 대주주 등과의 거래내용

[추출 결과 요약]

  [II] II. 사업의 내용
    문단 수: 133
    글자 수: 26,264
    앞 200자: 당사는 본사를 거점으로 한국과 DX 부문 산하 해외 9개 지역총괄 및 DS 부문 산하 해외 5개 지역총괄의 생산ㆍ판매법인, SDC 및 Harman 산하 종속기업 등 228개의 종속기업으로 구성된 글로벌 전자 기업입니다.
사업별로 보면, Set 사업은 DX(Device eXperience) 부문이 TV를 비롯하여 모니터, 냉장고, 세탁기, 에어컨, 스마트폰,

  [IV] IV. 이사의 경영진단 및 분석의견
    문단 수: 44
    글자 수: 10,863
    앞 200자: 1. 예측정보에 대한 주의사항 본 자료는 미래에 대한 '예측정보'를 포함하고 있습니다.이는 과거가 아닌 미래의 사건과 관계된 것으로 회사의 향후 예상되는 경영현황 및 재무실적을 의미하고, 표현상으로는 '예상', '전망', '계획', '기대' 등과 같은 단어를 포함합니다.'예측정보'는 그 성격상 불확실한 사건들을 언급하는데, 회사의 향후 경영현황 

### **1-9. 전체 파이프라인 — 381개 기업-연도 수집**

삼성전자 1개로 절차를 검증했다 (corp_code 매핑 → rcept_no 검색 →
원문 다운로드 → II/IV/VI 추출). 이제 동일 절차를 381개 전체에 적용한다.

**firm-year 문서 통합 (가이드 1단계):** 같은 기업-연도의 II/IV/VI 섹션
텍스트를 하나의 document로 묶는다. 회귀분석 표는 firm-year당 1행.

**수집 실패 처리 (가이드 원칙):**
- 보고서 못 찾음 / 원문 없음 / 추출 실패 행은 별도 기록
- 가짜 0으로 채우지 않음. 분석 corpus는 성공한 행만
- 실패 사유를 최종 보고서에 기록

**API 호출량:** 381행 × 2 호출(list + document) ≈ 762회.
호출 간 sleep으로 요청 제한(분당) 회피. 캐시로 재실행 시 0 호출.

In [20]:
def collect_one_firm_year(row, api_key, output_dir, sleep_sec=0.5):
    """
    한 firm-year 행을 받아 전체 수집 수행.
    
    Returns: dict (성공 시 텍스트 포함, 실패 시 fail_reason 포함)
    """
    stock_code = row["stock_code"]
    corp_code = row["corp_code"]
    fiscal_year = int(row["fiscal_year"])
    company_name = row["company_name"]
    
    base = {
        "company_name": company_name,
        "stock_code": stock_code,
        "corp_code": corp_code,
        "fiscal_year": fiscal_year,
        "esg_year": int(row["esg_year"]),
    }
    
    # --- 1) rcept_no 검색 ---
    try:
        rep = find_business_report(corp_code, fiscal_year, api_key, verbose=False)
    except Exception as e:
        return {**base, "status": "FAIL", "fail_reason": f"report_search_error: {e}"}
    
    if rep is None:
        return {**base, "status": "FAIL", "fail_reason": "no_business_report_found"}
    
    rcept_no = rep["rcept_no"]
    time.sleep(sleep_sec)
    
    # --- 2) 원문 다운로드 ---
    try:
        xml_text, _, chosen_name = download_document_xml(
            rcept_no, api_key, output_dir, verbose=False
        )
    except Exception as e:
        return {**base, "status": "FAIL", "rcept_no": rcept_no,
                "fail_reason": f"document_download_error: {e}"}
    
    # --- 3) II/IV/VI 추출 ---
    try:
        secs = extract_esg_sections(xml_text, verbose=False)
    except Exception as e:
        return {**base, "status": "FAIL", "rcept_no": rcept_no,
                "fail_reason": f"section_extract_error: {e}"}
    
    # 3개 섹션 중 하나도 못 뽑으면 실패
    if not secs or all(secs.get(r, {}).get("n_chars", 0) == 0 for r in ["II", "IV", "VI"]):
        return {**base, "status": "FAIL", "rcept_no": rcept_no,
                "fail_reason": "no_section_text_extracted"}
    
    # --- 4) firm-year 문서 통합 ---
    parts = []
    section_log = {}
    for roman in ["II", "IV", "VI"]:
        if roman in secs and secs[roman]["n_chars"] > 0:
            parts.append(secs[roman]["text"])
            section_log[roman] = secs[roman]["n_chars"]
        else:
            section_log[roman] = 0
    
    firm_year_doc = "\n".join(parts)
    
    return {
        **base,
        "status": "SUCCESS",
        "rcept_no": rcept_no,
        "report_nm": rep["report_nm"],
        "rcept_dt": rep["rcept_dt"],
        "n_report_candidates": rep["n_candidates"],
        "section_chars_II": section_log.get("II", 0),
        "section_chars_IV": section_log.get("IV", 0),
        "section_chars_VI": section_log.get("VI", 0),
        "total_chars": len(firm_year_doc),
        "firm_year_doc": firm_year_doc,
        "viewer_url": f"https://dart.fss.or.kr/dsaf001/main.do?rcpNo={rcept_no}",
    }


# 먼저 3개 행으로 소규모 테스트 (삼성전자 3개 연도)
print("[소규모 테스트 — 삼성전자 3개 연도]\n")
test_rows = cm[cm["stock_code"] == "005930"].to_dict("records")

for r in test_rows:
    out = collect_one_firm_year(r, API_KEY, OUTPUT_DIR, sleep_sec=0.5)
    print(f"  {out['company_name']} FY{out['fiscal_year']}: {out['status']}", end="")
    if out["status"] == "SUCCESS":
        print(f" | II={out['section_chars_II']:,} IV={out['section_chars_IV']:,} "
              f"VI={out['section_chars_VI']:,} | 총 {out['total_chars']:,}자")
    else:
        print(f" | 실패: {out['fail_reason']}")

[소규모 테스트 — 삼성전자 3개 연도]

  삼성전자 FY2022: SUCCESS | II=24,412 IV=12,261 VI=5,202 | 총 41,877자
  삼성전자 FY2023: SUCCESS | II=25,756 IV=11,972 VI=5,445 | 총 43,175자
  삼성전자 FY2024: SUCCESS | II=26,264 IV=10,863 VI=5,453 | 총 42,582자


### **1-10. 전체 381개 기업-연도 수집 실행**

소규모 테스트(삼성전자 3개 연도) 검증 완료. 동일 절차를 381개 전체에 적용.

**저장 구조 (메타/텍스트 분리):**
- `outputs/collection_meta.csv` — 메타데이터 (lineage 추적용)
- `outputs/corpus/{stock_code}_{fiscal_year}.json` — firm-year 텍스트

**체크포인트:** 행마다 메타 CSV 갱신. 중단 시 이미 SUCCESS 행은 스킵하고
재개. API 호출 간 0.5초 sleep (요청 제한 회피).

**수집 실패 처리:** 실패 행은 status=FAIL + fail_reason 기록.
가짜 0으로 채우지 않고 분석 corpus에서 제외 (가이드 원칙).

In [21]:
corpus_dir = os.path.join(OUTPUT_DIR, "corpus")
os.makedirs(corpus_dir, exist_ok=True)

meta_csv_path = os.path.join(OUTPUT_DIR, "collection_meta.csv")

# 메타 CSV에 저장할 컬럼 (firm_year_doc 텍스트는 제외 — 별도 JSON)
META_COLS = [
    "company_name", "stock_code", "corp_code", "fiscal_year", "esg_year",
    "status", "rcept_no", "report_nm", "rcept_dt", "n_report_candidates",
    "section_chars_II", "section_chars_IV", "section_chars_VI",
    "total_chars", "viewer_url", "fail_reason",
]

# 체크포인트 로드: 이미 수집된 메타가 있으면 SUCCESS 행 스킵
if os.path.exists(meta_csv_path):
    done_meta = pd.read_csv(meta_csv_path, dtype={"stock_code": str})
    done_meta["stock_code"] = done_meta["stock_code"].str.zfill(6)
    done_keys = set(
        zip(done_meta["stock_code"], done_meta["fiscal_year"])
    )
    success_keys = set(
        zip(
            done_meta.loc[done_meta["status"] == "SUCCESS", "stock_code"],
            done_meta.loc[done_meta["status"] == "SUCCESS", "fiscal_year"],
        )
    )
    print(f"[체크포인트 로드] 기존 {len(done_meta)}행, SUCCESS {len(success_keys)}개")
    meta_records = done_meta.to_dict("records")
else:
    success_keys = set()
    meta_records = []
    print("[체크포인트 없음 — 처음부터 수집]")


# 전체 행 순회
all_rows = cm.to_dict("records")
total = len(all_rows)
t_start = time.time()

for idx, row in enumerate(all_rows, 1):
    key = (row["stock_code"], int(row["fiscal_year"]))
    
    # 이미 SUCCESS면 스킵
    if key in success_keys:
        continue
    
    out = collect_one_firm_year(row, API_KEY, OUTPUT_DIR, sleep_sec=0.5)
    
    # 성공 시 텍스트를 JSON으로 저장 (메타와 분리)
    if out["status"] == "SUCCESS":
        doc_path = os.path.join(
            corpus_dir, f"{out['stock_code']}_{out['fiscal_year']}.json"
        )
        with open(doc_path, "w", encoding="utf-8") as f:
            json.dump({
                "stock_code": out["stock_code"],
                "fiscal_year": out["fiscal_year"],
                "esg_year": out["esg_year"],
                "rcept_no": out["rcept_no"],
                "firm_year_doc": out["firm_year_doc"],
            }, f, ensure_ascii=False)
    
    # 메타 레코드 (텍스트 제외)
    meta_records.append({c: out.get(c, None) for c in META_COLS})
    
    # 매 행마다 메타 CSV 갱신 (체크포인트)
    pd.DataFrame(meta_records).to_csv(meta_csv_path, index=False, encoding="utf-8-sig")
    
    # 진행 상황 (20행마다)
    if idx % 20 == 0 or idx == total:
        elapsed = time.time() - t_start
        n_succ = sum(1 for m in meta_records if m["status"] == "SUCCESS")
        n_fail = sum(1 for m in meta_records if m["status"] == "FAIL")
        print(f"  [{idx:>3}/{total}] 누적 성공 {n_succ} 실패 {n_fail} "
              f"| 경과 {elapsed/60:.1f}분")

print(f"\n[수집 완료] 총 소요: {(time.time()-t_start)/60:.1f}분")

[체크포인트 로드] 기존 381행, SUCCESS 381개

[수집 완료] 총 소요: 0.0분


In [22]:
# 최종 메타 로드
meta = pd.read_csv(meta_csv_path, dtype={"stock_code": str})
meta["stock_code"] = meta["stock_code"].str.zfill(6)

print("[수집 결과 요약]\n")
print(f"전체 행: {len(meta)}")
print(f"  SUCCESS: {(meta['status']=='SUCCESS').sum()}")
print(f"  FAIL:    {(meta['status']=='FAIL').sum()}")

# 연도별 성공률
print(f"\n[연도별 성공]")
for fy in sorted(meta["fiscal_year"].unique()):
    sub = meta[meta["fiscal_year"] == fy]
    n_succ = (sub["status"] == "SUCCESS").sum()
    print(f"  FY{fy}: {n_succ}/{len(sub)} 성공")

# 실패 사유 분류
fail = meta[meta["status"] == "FAIL"]
if len(fail) > 0:
    print(f"\n[실패 사유별 — 총 {len(fail)}건]")
    print(fail["fail_reason"].value_counts().to_string())
    print(f"\n[실패 기업-연도 목록]")
    print(fail[["company_name", "stock_code", "fiscal_year", "fail_reason"]]
          .to_string(index=False))
else:
    print(f"\n→ 전체 수집 성공, 실패 0건")

# 텍스트 분량 통계 (성공 행만)
succ = meta[meta["status"] == "SUCCESS"]
print(f"\n[firm-year 문서 분량 통계 — 성공 {len(succ)}행]")
print(f"  total_chars 평균: {succ['total_chars'].mean():,.0f}")
print(f"  total_chars 최소: {succ['total_chars'].min():,}")
print(f"  total_chars 최대: {succ['total_chars'].max():,}")
print(f"  total_chars 중앙값: {succ['total_chars'].median():,.0f}")

# 분량 너무 적은 행 점검 (추출 부실 의심)
thin = succ[succ["total_chars"] < 1000]
if len(thin) > 0:
    print(f"\n[⚠️ 분량 1000자 미만 — 추출 부실 의심 {len(thin)}건]")
    print(thin[["company_name", "stock_code", "fiscal_year", 
                "section_chars_II", "section_chars_IV", "section_chars_VI"]]
          .to_string(index=False))

[수집 결과 요약]

전체 행: 381
  SUCCESS: 381
  FAIL:    0

[연도별 성공]
  FY2022: 127/127 성공
  FY2023: 127/127 성공
  FY2024: 127/127 성공

→ 전체 수집 성공, 실패 0건

[firm-year 문서 분량 통계 — 성공 381행]
  total_chars 평균: 34,222
  total_chars 최소: 4,194.0
  total_chars 최대: 186,911.0
  total_chars 중앙값: 25,520


### **1-11. 수집 실패 13건 원인 진단**

실패는 두 유형:
- `no_business_report_found` (6건): 신영증권·만호제강 전체 연도
  → 결산월이 12월이 아닐 가능성 (현재 코드는 `(YYYY.12)` 필터)
- `document_download_error: not a zip` (7건): rcept_no는 찾았으나
  원문이 ZIP이 아님 → 응답 내용 직접 확인 필요

각 유형의 실제 원인을 진단한 뒤, 살릴 수 있는 행을 재수집한다.

In [23]:
# 신영증권, 만호제강이 어떤 보고서를 제출했는지 직접 확인
# (필터 없이 모든 사업보고서 조회)

def diagnose_no_report(corp_code, company_name, fiscal_year, api_key):
    """검색 기간 내 모든 사업보고서를 필터 없이 조회 (결산월 확인)"""
    search_year = fiscal_year + 1
    params = {
        "crtfc_key": api_key,
        "corp_code": corp_code,
        "bgn_de": f"{search_year}0101",
        "end_de": f"{search_year}1231",
        "pblntf_detail_ty": "A001",
        "page_count": "100",
    }
    res = requests.get(
        "https://opendart.fss.or.kr/api/list.json",
        params=params, timeout=20,
    )
    payload = res.json()
    
    print(f"\n[{company_name} ({corp_code}) FY{fiscal_year} → 검색 {search_year}년]")
    print(f"  API status: {payload.get('status')} ({payload.get('message')})")
    
    rows = payload.get("list", [])
    print(f"  전체 사업보고서류 {len(rows)}개:")
    for r in rows:
        print(f"    - {r.get('report_nm')} | rcept={r.get('rcept_no')} | {r.get('rcept_dt')}")
    return rows


# 신영증권, 만호제강 진단
diag_targets = [
    ("신영증권", "001720"),
    ("만호제강", "001080"),
]

for name, scode in diag_targets:
    ccode = cm[cm["stock_code"] == scode]["corp_code"].iloc[0]
    # 2024 연도만 대표로 진단
    diagnose_no_report(ccode, name, 2024, API_KEY)
    time.sleep(0.5)


[신영증권 (00136721) FY2024 → 검색 2025년]
  API status: 000 (정상)
  전체 사업보고서류 2개:
    - [기재정정]사업보고서 (2025.03) | rcept=20250829000681 | 20250829
    - 사업보고서 (2025.03) | rcept=20250612000448 | 20250612

[만호제강 (00120872) FY2024 → 검색 2025년]
  API status: 000 (정상)
  전체 사업보고서류 4개:
    - [기재정정]사업보고서 (2024.06) | rcept=20250930000631 | 20250930
    - [기재정정]사업보고서 (2025.06) | rcept=20250918000340 | 20250918
    - 사업보고서 (2025.06) | rcept=20250918000297 | 20250918
    - [기재정정]사업보고서 (2024.06) | rcept=20250805000299 | 20250805


In [24]:
# document_download_error 난 기업들의 실제 응답 확인
def diagnose_not_zip(corp_code, company_name, fiscal_year, api_key):
    """rcept_no를 찾고, document.xml 응답이 무엇인지 직접 확인"""
    rep = find_business_report(corp_code, fiscal_year, api_key, verbose=False)
    if rep is None:
        print(f"\n[{company_name} FY{fiscal_year}] rcept_no 못 찾음")
        return
    
    rcept_no = rep["rcept_no"]
    res = requests.get(
        "https://opendart.fss.or.kr/api/document.xml",
        params={"crtfc_key": api_key, "rcept_no": rcept_no},
        timeout=60,
    )
    
    print(f"\n[{company_name} FY{fiscal_year}] rcept_no={rcept_no}")
    print(f"  report_nm: {rep['report_nm']}")
    print(f"  HTTP: {res.status_code}")
    print(f"  Content-Type: {res.headers.get('Content-Type')}")
    print(f"  응답 크기: {len(res.content):,} bytes")
    
    # ZIP 시그니처 확인 (ZIP은 'PK'로 시작)
    head = res.content[:4]
    print(f"  앞 4바이트: {head}")
    
    if head[:2] == b'PK':
        print(f"  → ZIP 맞음 (재시도로 해결 가능)")
    else:
        # ZIP 아니면 텍스트로 디코딩해서 내용 확인 (보통 에러 XML/JSON)
        try:
            txt = res.content[:500].decode("utf-8", errors="ignore")
            print(f"  → ZIP 아님. 응답 앞 500자:\n{txt}")
        except Exception as e:
            print(f"  → 디코딩 실패: {e}")


not_zip_targets = [
    ("CJ대한통운", "000120", 2024),
    ("우리금융지주", "316140", 2024),
    ("유수홀딩스", "000700", 2024),
    ("이수화학", "005950", 2022),
    ("CS홀딩스", "000590", 2023),
    ("전방", "000950", 2024),
]

for name, scode, fy in not_zip_targets:
    ccode = cm[cm["stock_code"] == scode]["corp_code"].iloc[0]
    diagnose_not_zip(ccode, name, fy, API_KEY)
    time.sleep(0.5)


[CJ대한통운 FY2024] rcept_no=20250317001020
  report_nm: [첨부정정]사업보고서 (2024.12)
  HTTP: 200
  Content-Type: application/xml;charset=UTF-8
  응답 크기: 147 bytes
  앞 4바이트: b'<?xm'
  → ZIP 아님. 응답 앞 500자:
<?xml version="1.0" encoding="UTF-8" standalone="yes"?><result><status>014</status><message>파일이 존재하지 않습니다.</message></result>

[우리금융지주 FY2024] rcept_no=20250318001303
  report_nm: [첨부정정]사업보고서 (2024.12)
  HTTP: 200
  Content-Type: application/xml;charset=UTF-8
  응답 크기: 147 bytes
  앞 4바이트: b'<?xm'
  → ZIP 아님. 응답 앞 500자:
<?xml version="1.0" encoding="UTF-8" standalone="yes"?><result><status>014</status><message>파일이 존재하지 않습니다.</message></result>

[유수홀딩스 FY2024] rcept_no=20250331003248
  report_nm: [첨부정정]사업보고서 (2024.12)
  HTTP: 200
  Content-Type: application/xml;charset=UTF-8
  응답 크기: 147 bytes
  앞 4바이트: b'<?xm'
  → ZIP 아님. 응답 앞 500자:
<?xml version="1.0" encoding="UTF-8" standalone="yes"?><result><status>014</status><message>파일이 존재하지 않습니다.</message></result>

[이수화학 FY2022] rcept_no=20230328001079
  

### **1-12. 수집 함수 수정 — 결산월 유연 + 정정 회피**

진단 결과 두 가지 버그 발견:

1. **`.12` 하드코딩**: `(fiscal_year.12)` 만 검색 → 3월·6월 결산 기업
   (신영증권, 만호제강) 누락. 가이드 문구("fiscal_year+1년 공시
   사업보고서")에 따라 결산월 무관하게 매칭하도록 수정.

2. **`[첨부정정]` 함정**: 가장 최근 접수를 골랐는데 `[첨부정정]`이면
   document.xml에 원문이 없음(status 014). 원문이 존재하는 버전을
   우선 선택하도록 정정 우선순위 재설계.

**선택 우선순위:** 원본 사업보고서 > `[기재정정]` > `[첨부정정]`
같은 종류 내에서는 가장 최근 접수.

**비12월 결산 처리 한계:** fiscal_year+1년 공시 사업보고서를 매칭하므로
회계기간이 12월 결산 기업과 정확히 일치하지 않는다. 과제의 타이밍 규칙
(KCGS 평가연도 ↔ 직전 회계연도 보고서)에는 부합. 보고서에 명시.

In [25]:
def find_business_report_v2(corp_code, fiscal_year, api_key, verbose=True):
    """
    fiscal_year+1년에 공시된 사업보고서를 찾는다 (결산월 무관).
    
    수정 사항:
    - .12 하드코딩 제거 → "사업보고서" 포함이면 결산월 무관 매칭
    - [첨부정정] 회피 → 원문 있는 버전 우선
    - 정정 우선순위: 원본 > [기재정정] > [첨부정정]
    
    Returns: dict 또는 None
    """
    search_year = fiscal_year + 1
    params = {
        "crtfc_key": api_key,
        "corp_code": corp_code,
        "bgn_de": f"{search_year}0101",
        "end_de": f"{search_year}1231",
        "pblntf_detail_ty": "A001",
        "page_count": "100",
    }
    res = requests.get(
        "https://opendart.fss.or.kr/api/list.json",
        params=params, timeout=20,
    )
    payload = res.json()
    
    if payload.get("status") != "000":
        if verbose:
            print(f"  status={payload.get('status')}: {payload.get('message')}")
        return None
    
    rows = payload.get("list", [])
    
    # "사업보고서" 포함된 것만 (결산월 무관) — 반기/분기 제외
    candidates = [
        r for r in rows
        if "사업보고서" in r.get("report_nm", "")
        and "반기" not in r.get("report_nm", "")
        and "분기" not in r.get("report_nm", "")
    ]
    
    if verbose:
        print(f"  검색 {search_year}년, '사업보고서' 후보 {len(candidates)}개:")
        for c in candidates:
            print(f"    - {c.get('report_nm')} | {c.get('rcept_no')} | {c.get('rcept_dt')}")
    
    if not candidates:
        return None
    
    # 정정 종류로 우선순위 부여 (낮을수록 우선)
    def priority(report_nm):
        if "[첨부정정]" in report_nm:
            return 2   # 원문 없을 수 있음 → 최후
        elif "[기재정정]" in report_nm or "[정정]" in report_nm:
            return 1   # 본문 정정 → 차선
        else:
            return 0   # 원본 → 최우선 (원문 확실)
    
    # (우선순위 오름차순, 접수일 내림차순) 정렬
    candidates_sorted = sorted(
        candidates,
        key=lambda r: (priority(r.get("report_nm", "")), 
                       -int(r.get("rcept_dt", "0"))),
    )
    chosen = candidates_sorted[0]
    
    return {
        "corp_code": corp_code,
        "fiscal_year": fiscal_year,
        "report_nm": chosen.get("report_nm"),
        "rcept_no": chosen.get("rcept_no"),
        "rcept_dt": chosen.get("rcept_dt"),
        "n_candidates": len(candidates),
        "all_report_nms": [c.get("report_nm") for c in candidates],
        "selection_priority": priority(chosen.get("report_nm", "")),
    }


# 13건 재수집 대상
refail_targets = [
    ("신영증권", "001720"), ("만호제강", "001080"),
    ("CJ대한통운", "000120"), ("우리금융지주", "316140"),
    ("유수홀딩스", "000700"), ("이수화학", "005950"),
    ("CS홀딩스", "000590"), ("전방", "000950"),
]

# 검증: 수정 함수로 다시 rcept_no 찾기 (다운로드 전 확인)
print("[수정 함수로 rcept_no 재탐색]\n")
for name, scode in refail_targets:
    ccode = cm[cm["stock_code"] == scode]["corp_code"].iloc[0]
    fail_fys = meta[(meta["stock_code"] == scode) & 
                    (meta["status"] == "FAIL")]["fiscal_year"].tolist()
    for fy in sorted(fail_fys):
        print(f"[{name} FY{fy}]")
        rep = find_business_report_v2(ccode, int(fy), API_KEY, verbose=True)
        if rep:
            print(f"  → 선택: {rep['report_nm']} (priority={rep['selection_priority']})")
        else:
            print(f"  → 여전히 못 찾음")
        print()
        time.sleep(0.5)

[수정 함수로 rcept_no 재탐색]



### **1-13. 실패 13건 재수집 및 corpus 통합**

수정 함수(`find_business_report_v2`)로 13건 전부 원본 사업보고서
(priority=0) 재매칭 확인. 이제 실제 다운로드 + II/IV/VI 추출 +
corpus/메타 갱신.

성공 시 `collection_meta.csv`의 해당 행을 FAIL → SUCCESS로 갱신하고
corpus JSON을 추가한다.

In [26]:
def collect_one_firm_year_v2(row, api_key, output_dir, sleep_sec=0.5):
    """find_business_report_v2 사용 버전 (결산월 유연 + 정정 회피)"""
    stock_code = row["stock_code"]
    corp_code = row["corp_code"]
    fiscal_year = int(row["fiscal_year"])
    company_name = row["company_name"]
    
    base = {
        "company_name": company_name,
        "stock_code": stock_code,
        "corp_code": corp_code,
        "fiscal_year": fiscal_year,
        "esg_year": int(row["esg_year"]),
    }
    
    try:
        rep = find_business_report_v2(corp_code, fiscal_year, api_key, verbose=False)
    except Exception as e:
        return {**base, "status": "FAIL", "fail_reason": f"report_search_error: {e}"}
    
    if rep is None:
        return {**base, "status": "FAIL", "fail_reason": "no_business_report_found"}
    
    rcept_no = rep["rcept_no"]
    time.sleep(sleep_sec)
    
    try:
        xml_text, _, _ = download_document_xml(rcept_no, api_key, output_dir, verbose=False)
    except Exception as e:
        return {**base, "status": "FAIL", "rcept_no": rcept_no,
                "fail_reason": f"document_download_error: {e}"}
    
    try:
        secs = extract_esg_sections(xml_text, verbose=False)
    except Exception as e:
        return {**base, "status": "FAIL", "rcept_no": rcept_no,
                "fail_reason": f"section_extract_error: {e}"}
    
    if not secs or all(secs.get(r, {}).get("n_chars", 0) == 0 for r in ["II", "IV", "VI"]):
        return {**base, "status": "FAIL", "rcept_no": rcept_no,
                "fail_reason": "no_section_text_extracted"}
    
    parts, section_log = [], {}
    for roman in ["II", "IV", "VI"]:
        if roman in secs and secs[roman]["n_chars"] > 0:
            parts.append(secs[roman]["text"])
            section_log[roman] = secs[roman]["n_chars"]
        else:
            section_log[roman] = 0
    
    firm_year_doc = "\n".join(parts)
    
    return {
        **base,
        "status": "SUCCESS",
        "rcept_no": rcept_no,
        "report_nm": rep["report_nm"],
        "rcept_dt": rep["rcept_dt"],
        "n_report_candidates": rep["n_candidates"],
        "section_chars_II": section_log.get("II", 0),
        "section_chars_IV": section_log.get("IV", 0),
        "section_chars_VI": section_log.get("VI", 0),
        "total_chars": len(firm_year_doc),
        "firm_year_doc": firm_year_doc,
        "viewer_url": f"https://dart.fss.or.kr/dsaf001/main.do?rcpNo={rcept_no}",
    }


# 메타 로드 + FAIL 행만 재수집
meta = pd.read_csv(meta_csv_path, dtype={"stock_code": str})
meta["stock_code"] = meta["stock_code"].str.zfill(6)

fail_rows = meta[meta["status"] == "FAIL"].copy()
print(f"[재수집 대상: {len(fail_rows)}건]\n")

recollect_results = []
for _, frow in fail_rows.iterrows():
    # cm에서 원본 행 정보 가져오기
    orig = cm[(cm["stock_code"] == frow["stock_code"]) &
              (cm["fiscal_year"] == frow["fiscal_year"])].iloc[0].to_dict()
    
    out = collect_one_firm_year_v2(orig, API_KEY, OUTPUT_DIR, sleep_sec=0.5)
    recollect_results.append(out)
    
    status_mark = "✓" if out["status"] == "SUCCESS" else "✗"
    print(f"  {status_mark} {out['company_name']} FY{out['fiscal_year']}: {out['status']}", end="")
    if out["status"] == "SUCCESS":
        print(f" | II={out['section_chars_II']:,} IV={out['section_chars_IV']:,} "
              f"VI={out['section_chars_VI']:,} | 총 {out['total_chars']:,}자")
    else:
        print(f" | {out['fail_reason']}")

n_recovered = sum(1 for r in recollect_results if r["status"] == "SUCCESS")
print(f"\n[재수집 결과] {n_recovered}/{len(fail_rows)}건 복구")

[재수집 대상: 0건]


[재수집 결과] 0/0건 복구


In [27]:
# 메타 전체를 문자열로 로드
meta = pd.read_csv(meta_csv_path, dtype=str, keep_default_na=False)
meta["stock_code"] = meta["stock_code"].str.zfill(6)

META_COLS = [
    "company_name", "stock_code", "corp_code", "fiscal_year", "esg_year",
    "status", "rcept_no", "report_nm", "rcept_dt", "n_report_candidates",
    "section_chars_II", "section_chars_IV", "section_chars_VI",
    "total_chars", "viewer_url", "fail_reason",
]

# 재수집 결과를 딕셔너리로 (키: stock_code_fiscal_year)
recollect_map = {}
for out in recollect_results:
    if out["status"] == "SUCCESS":
        key = f"{out['stock_code']}_{int(out['fiscal_year'])}"
        recollect_map[key] = out
        
        # corpus JSON 저장 (없으면)
        doc_path = os.path.join(corpus_dir, f"{key}.json")
        if not os.path.exists(doc_path):
            with open(doc_path, "w", encoding="utf-8") as f:
                json.dump({
                    "stock_code": out["stock_code"],
                    "fiscal_year": int(out["fiscal_year"]),
                    "esg_year": int(out["esg_year"]),
                    "rcept_no": out["rcept_no"],
                    "firm_year_doc": out["firm_year_doc"],
                }, f, ensure_ascii=False)

print(f"재수집 성공 맵: {len(recollect_map)}건")

# 메타를 dict 리스트로 변환 후, 해당 행을 교체해서 새 DataFrame 구성
records = meta.to_dict("records")

n_updated = 0
for rec in records:
    key = f"{rec['stock_code'].zfill(6)}_{int(rec['fiscal_year'])}"
    if key in recollect_map:
        out = recollect_map[key]
        for col in META_COLS:
            if col in out and out.get(col) is not None:
                rec[col] = str(out.get(col))   # dict라 dtype 충돌 없음
        n_updated += 1

# 새 DataFrame (전부 문자열) — dtype 충돌 원천 차단
meta_new = pd.DataFrame(records, columns=meta.columns).astype(str)
meta_new.to_csv(meta_csv_path, index=False, encoding="utf-8-sig")

print(f"메타 갱신 완료: {n_updated}건 업데이트\n")

# 최종 요약
print("[최종 수집 결과]")
print(f"전체: {len(meta_new)}")
print(f"  SUCCESS: {(meta_new['status']=='SUCCESS').sum()}")
print(f"  FAIL:    {(meta_new['status']=='FAIL').sum()}")

print(f"\n[연도별 성공]")
for fy in sorted(meta_new["fiscal_year"].unique()):
    sub = meta_new[meta_new["fiscal_year"] == fy]
    print(f"  FY{fy}: {(sub['status']=='SUCCESS').sum()}/{len(sub)}")

still_fail = meta_new[meta_new["status"] == "FAIL"]
if len(still_fail) > 0:
    print(f"\n[남은 실패 {len(still_fail)}건]")
    print(still_fail[["company_name", "stock_code", "fiscal_year", "fail_reason"]]
          .to_string(index=False))
else:
    print(f"\n→ 전체 381건 수집 성공 (실패 0) 🎉")

corpus_files = [f for f in os.listdir(corpus_dir) if f.endswith(".json")]
print(f"\ncorpus JSON 파일 수: {len(corpus_files)}")

재수집 성공 맵: 0건
메타 갱신 완료: 0건 업데이트

[최종 수집 결과]
전체: 381
  SUCCESS: 381
  FAIL:    0

[연도별 성공]
  FY2022: 127/127
  FY2023: 127/127
  FY2024: 127/127

→ 전체 381건 수집 성공 (실패 0) 🎉

corpus JSON 파일 수: 381


1. 수집 도구: OpenDART API 직접 (재현성 최고)
2. corp_code 매핑: corpCode.xml 전체 다운로드 (127개 1회 매핑)
3. 타이밍: esg_year = fiscal_year + 1, fiscal_year+1년 공시 사업보고서
4. 비12월 결산: 신영증권(3월)/만호제강(6월) → 회계기간 불일치 한계 명시
5. 정정공시: 원본 > [기재정정] > [첨부정정] 우선순위
   ([첨부정정]은 document.xml 원문 없음 — status 014)
6. 섹션 선택: II(사업의 내용), IV(경영진단), VI(이사회) — ESG 텍스트 집중
7. 실패 처리: 13건 원인 진단 후 전수 복구 (가짜 0 없음)

---
## **Phase 2: 한국어 텍스트 전처리**

수집한 381개 firm-year 문서를 형태소 분석으로 토큰화하고 불용어를
제거한다. 이 토큰들이 Phase 3의 TF-IDF feature 입력이 된다.

### **2-1. corpus 로드 및 기초 통계**

`outputs/corpus/{stock_code}_{fiscal_year}.json` 381개를 로드하고
문서 길이 분포를 확인한다. 이상치(너무 짧거나 긴 문서)를 점검한다.

In [28]:
import glob

# corpus JSON 전체 로드
corpus_files = sorted(glob.glob(os.path.join(corpus_dir, "*.json")))
print(f"corpus 파일 수: {len(corpus_files)}")

corpus_records = []
for fp in corpus_files:
    with open(fp, "r", encoding="utf-8") as f:
        d = json.load(f)
    corpus_records.append({
        "stock_code": str(d["stock_code"]).zfill(6),
        "fiscal_year": int(d["fiscal_year"]),
        "esg_year": int(d["esg_year"]),
        "rcept_no": d["rcept_no"],
        "doc": d["firm_year_doc"],
        "doc_len": len(d["firm_year_doc"]),
    })

corpus_df = pd.DataFrame(corpus_records)
print(f"corpus_df shape: {corpus_df.shape}")

# 메타와 일치 확인
meta = pd.read_csv(meta_csv_path, dtype=str, keep_default_na=False)
meta["stock_code"] = meta["stock_code"].str.zfill(6)
meta_success = meta[meta["status"] == "SUCCESS"]
print(f"메타 SUCCESS: {len(meta_success)} | corpus: {len(corpus_df)}")
assert len(corpus_df) == 381, "corpus 수 불일치"

# 문서 길이 통계
print(f"\n[문서 길이(글자 수) 통계]")
print(f"  평균:   {corpus_df['doc_len'].mean():,.0f}")
print(f"  중앙값: {corpus_df['doc_len'].median():,.0f}")
print(f"  최소:   {corpus_df['doc_len'].min():,}")
print(f"  최대:   {corpus_df['doc_len'].max():,}")
print(f"  표준편차: {corpus_df['doc_len'].std():,.0f}")

# 분위수
print(f"\n[분위수]")
for q in [0.05, 0.25, 0.5, 0.75, 0.95]:
    print(f"  {int(q*100)}%: {corpus_df['doc_len'].quantile(q):,.0f}")

# 짧은 문서 점검 (하위 5개)
print(f"\n[가장 짧은 문서 5개]")
short5 = corpus_df.nsmallest(5, "doc_len")
for _, r in short5.iterrows():
    name = meta[(meta["stock_code"]==r["stock_code"]) & 
                (meta["fiscal_year"].astype(int)==r["fiscal_year"])]["company_name"].iloc[0]
    print(f"  {name} ({r['stock_code']}) FY{r['fiscal_year']}: {r['doc_len']:,}자")

# 긴 문서 (상위 5개)
print(f"\n[가장 긴 문서 5개]")
long5 = corpus_df.nlargest(5, "doc_len")
for _, r in long5.iterrows():
    name = meta[(meta["stock_code"]==r["stock_code"]) & 
                (meta["fiscal_year"].astype(int)==r["fiscal_year"])]["company_name"].iloc[0]
    print(f"  {name} ({r['stock_code']}) FY{r['fiscal_year']}: {r['doc_len']:,}자")

corpus 파일 수: 381
corpus_df shape: (381, 6)
메타 SUCCESS: 381 | corpus: 381

[문서 길이(글자 수) 통계]
  평균:   34,222
  중앙값: 25,520
  최소:   4,194
  최대:   186,911
  표준편차: 30,877

[분위수]
  5%: 9,620
  25%: 15,457
  50%: 25,520
  75%: 41,687
  95%: 104,076

[가장 짧은 문서 5개]
  천일고속 (000650) FY2024: 4,194자
  천일고속 (000650) FY2022: 4,369자
  천일고속 (000650) FY2023: 4,371자
  조흥 (002600) FY2024: 5,658자
  조흥 (002600) FY2023: 5,715자

[가장 긴 문서 5개]
  POSCO홀딩스 (005490) FY2022: 186,911자
  POSCO홀딩스 (005490) FY2023: 182,269자
  한화 (000880) FY2024: 179,335자
  POSCO홀딩스 (005490) FY2024: 171,742자
  CJ (001040) FY2023: 164,975자


### **2-2. 형태소 분석 — Kiwi 명사 토큰화**

형태소 분석기로 Kiwi를 사용한다 (C++ 기반으로 빠르고, 최신 사전 +
신조어 대응). seed/expanded dictionary가 모두 명사이므로 **명사만**
추출한다 (NNG 일반명사, NNP 고유명사).

**명사 추출 근거:**
- 30개 seed 단어 전부 명사 (탄소, 이사회, 공급망...)
- TF-IDF 측정 목적 = ESG 명사 빈도. 형용사/동사는 noise
- cheap-talk 측정 시 실질 키워드 집중

**길이 정규화 고려:** 문서 길이 편차 45배(4천~19만 자). Phase 3에서
total_word_count 통제 변수로 분리 예정.

In [29]:
from kiwipiepy import Kiwi

# Kiwi 인스턴스 (재현성: 기본 모델 고정)
kiwi = Kiwi()
print("Kiwi 로드 성공")

# 버전 확인
import kiwipiepy
print(f"kiwipiepy 버전: {kiwipiepy.__version__}")

# 토큰화 테스트 — 샘플 문서로
sample = corpus_df.iloc[0]
sample_text = sample["doc"][:300]
print(f"\n[샘플 원문 300자]")
print(sample_text)

# 형태소 분석 결과 확인 (품사 태그 보기)
tokens = kiwi.tokenize(sample_text)
print(f"\n[형태소 분석 결과 — 앞 20개]")
for t in tokens[:20]:
    print(f"  {t.form:>12} | {t.tag}")

Kiwi 로드 성공
kiwipiepy 버전: 0.23.1

[샘플 원문 300자]
1. 일반적인 사항지배기업인 연결실체는 제공하는 재화나 용역에 근거하여 영업부문을 구분하고 각 부문의 재무정보를 내부관리 목적으로 활용하고 있는 바, 당사는 제약업, 의료기기 제조 및 판매업, 기타금융업을 운영하고 있습니다. 영위하는 영업부문 중 기타금융업의 경우 중요성의 관점에서 비중이 크지 않습니다.
2. 지배회사의 현황(1) 영업개황 및 사업부문의 구분(연결기준) 당사의 연결실체는 94기 매출액은 340,426백만원(연결기준)으로 전년 동기 대비 16.2% 증가하였으며 영업이익은 29,915백만원(연결기준)으로 전년 동기 대비

[형태소 분석 결과 — 앞 20개]
            1. | SB
            일반 | NNG
             적 | XSN
             이 | VCP
             ᆫ | ETM
            사항 | NNG
            지배 | NNG
            기업 | NNG
             이 | VCP
             ᆫ | ETM
            연결 | NNG
            실체 | NNG
             는 | JX
            제공 | NNG
             하 | XSV
             는 | ETM
            재화 | NNG
             나 | JC
            용역 | NNG
             에 | JKB


In [30]:
def extract_nouns(text, kiwi_inst, min_len=2):
    """
    Kiwi로 명사(NNG 일반명사, NNP 고유명사)만 추출.
    
    - min_len: 최소 글자 수 (1글자 명사는 noise 많아 기본 2)
    - 숫자/영문 단독은 제외
    """
    tokens = kiwi_inst.tokenize(text)
    nouns = []
    for t in tokens:
        # NNG: 일반명사, NNP: 고유명사
        if t.tag in ("NNG", "NNP"):
            form = t.form.strip()
            # 최소 길이 + 숫자만/영문만 토큰 제외
            if len(form) >= min_len and not form.isdigit():
                nouns.append(form)
    return nouns


# 샘플로 명사 추출 테스트
sample_nouns = extract_nouns(sample["doc"][:500], kiwi)
print(f"[샘플 500자 → 명사 추출]")
print(f"명사 수: {len(sample_nouns)}")
print(f"앞 30개: {sample_nouns[:30]}")

# ESG seed 단어가 실제로 명사로 추출되는지 검증
test_sentence = "당사는 온실가스 배출량을 감축하고 재생에너지 사용을 확대하며 이사회 산하 감사위원회의 독립성을 강화하였습니다"
test_nouns = extract_nouns(test_sentence, kiwi)
print(f"\n[ESG 문장 테스트]")
print(f"원문: {test_sentence}")
print(f"명사: {test_nouns}")
print(f"\n→ seed 단어(온실가스, 재생에너지, 이사회, 감사위원회, 독립성)가 추출되는지 확인")

[샘플 500자 → 명사 추출]
명사 수: 109
앞 30개: ['일반', '사항', '지배', '기업', '연결', '실체', '제공', '재화', '용역', '근거', '영업', '부문', '구분', '부문', '재무', '정보', '내부', '관리', '목적', '활용', '당사', '제약', '의료', '기기', '제조', '판매업', '기타', '금융업', '운영', '영위']

[ESG 문장 테스트]
원문: 당사는 온실가스 배출량을 감축하고 재생에너지 사용을 확대하며 이사회 산하 감사위원회의 독립성을 강화하였습니다
명사: ['당사', '온실가스', '배출량', '감축', '재생', '에너지', '사용', '확대', '이사회', '산하', '감사', '위원회', '독립', '강화']

→ seed 단어(온실가스, 재생에너지, 이사회, 감사위원회, 독립성)가 추출되는지 확인


### **2-3. 복합명사 보존 — seed 사용자 사전 등록**

Kiwi 기본 분석은 복합명사를 분리한다 (`재생에너지` → `재생`+`에너지`,
`감사위원회` → `감사`+`위원회`). seed/expanded dictionary 단어가
분리되면 TF-IDF 매칭이 불가능하다.

**해결:** seed_dictionary의 모든 표현을 Kiwi 사용자 사전에 명사(NNP)로
등록해 한 토큰으로 보존한다. 이는 ESG 사전 매칭의 전제 조건이다.

**한계 기록:** 사용자 사전 등록은 측정 대상 단어를 우선 보존하는 분석적
선택이다. expanded dictionary 확장 시에도 동일 원칙을 적용한다.

In [31]:
# seed_dictionary 로드
seed_df = pd.read_csv(os.path.join(DATA_DIR, "seed_dictionary.csv"))
print(f"seed_dictionary shape: {seed_df.shape}")
print(f"컬럼: {list(seed_df.columns)}")
print(f"\n[차원별 seed 단어]")
for dim in ["E", "S", "G"]:
    terms = seed_df[seed_df["dimension"] == dim]["seed_term"].tolist()
    print(f"  {dim} ({len(terms)}개): {terms}")

# seed_term 전체 목록
seed_terms = seed_df["seed_term"].tolist()
print(f"\n총 seed 단어: {len(seed_terms)}개")

# pattern 컬럼도 확인 (정규식 패턴 — 동의어 포함)
print(f"\n[pattern 예시 — 동의어 포함된 것]")
for _, r in seed_df.head(10).iterrows():
    if "|" in str(r["pattern"]):
        print(f"  {r['seed_term']:>10} → pattern: {r['pattern']}")

seed_dictionary shape: (30, 6)
컬럼: ['dimension', 'seed_term', 'pattern', 'source_basis', 'source_titles', 'notes']

[차원별 seed 단어]
  E (10개): ['탄소', '온실가스', '탄소중립', '넷제로', '재생에너지', '에너지', '전력', '폐기물', '재활용', '폐수']
  S (10개): ['안전', '산업재해', '중대재해', '임직원', '노동', '인권', '교육훈련', '협력사', '공급망', '지역사회']
  G (10개): ['이사회', '사외이사', '감사위원회', '독립성', '윤리', '준법', '컴플라이언스', '부패방지', '주주', '의결권']

총 seed 단어: 30개

[pattern 예시 — 동의어 포함된 것]
        온실가스 → pattern: 온실가스|GHG
         넷제로 → pattern: 넷제로|net zero|net-zero
       재생에너지 → pattern: 재생에너지|renewable energy
          전력 → pattern: 전력|전력사용량
         재활용 → pattern: 재활용|자원순환
          폐수 → pattern: 폐수|수질|물관리


In [32]:
# Kiwi 재생성 (사용자 사전 깨끗하게)
kiwi = Kiwi()

# seed_term 전체를 사용자 사전에 NNP(고유명사)로 등록
# → 분리되지 않고 한 토큰으로 인식됨
n_added = 0
for term in seed_terms:
    term = str(term).strip()
    if term:
        kiwi.add_user_word(term, "NNP")
        n_added += 1

print(f"사용자 사전 등록: {n_added}개 seed 단어")

# 재검증 — 같은 ESG 문장
test_sentence = "당사는 온실가스 배출량을 감축하고 재생에너지 사용을 확대하며 이사회 산하 감사위원회의 독립성을 강화하였습니다"
test_nouns = extract_nouns(test_sentence, kiwi)
print(f"\n[ESG 문장 재검증]")
print(f"원문: {test_sentence}")
print(f"명사: {test_nouns}")

# seed 단어별 매칭 확인
print(f"\n[seed 단어 보존 여부]")
check_seeds = ["온실가스", "재생에너지", "이사회", "감사위원회", "독립성"]
for s in check_seeds:
    status = "✓ 보존" if s in test_nouns else "✗ 분리/누락"
    print(f"  {s}: {status}")

# 전체 30개 seed가 단일 토큰으로 인식되는지 일괄 검증
print(f"\n[전체 seed 30개 토큰화 검증]")
broken = []
for term in seed_terms:
    term = str(term).strip()
    toks = extract_nouns(term, kiwi)
    # term 자체가 그대로 하나의 토큰으로 나와야 정상
    if term not in toks:
        broken.append((term, toks))

if broken:
    print(f"  ⚠️ 여전히 분리되는 seed {len(broken)}개:")
    for term, toks in broken:
        print(f"    {term} → {toks}")
else:
    print(f"  ✓ 30개 seed 전부 단일 토큰으로 보존됨")

사용자 사전 등록: 30개 seed 단어

[ESG 문장 재검증]
원문: 당사는 온실가스 배출량을 감축하고 재생에너지 사용을 확대하며 이사회 산하 감사위원회의 독립성을 강화하였습니다
명사: ['당사', '온실가스', '배출량', '감축', '재생', '에너지', '사용', '확대', '이사회', '산하', '감사위원회', '독립', '강화']

[seed 단어 보존 여부]
  온실가스: ✓ 보존
  재생에너지: ✗ 분리/누락
  이사회: ✓ 보존
  감사위원회: ✓ 보존
  독립성: ✗ 분리/누락

[전체 seed 30개 토큰화 검증]
  ✓ 30개 seed 전부 단일 토큰으로 보존됨


In [33]:
# Kiwi 재생성
kiwi = Kiwi()

# seed 단어를 높은 score로 등록 → 문맥 무관 우선 인식
# score 기본 0 → 높게(예: 50) 주면 분석 시 이 단어 강제 우선
n_added = 0
for term in seed_terms:
    term = str(term).strip()
    if term:
        kiwi.add_user_word(term, "NNP", score=50.0)
        n_added += 1

print(f"사용자 사전 등록 (score=50): {n_added}개")

# 재검증
test_sentence = "당사는 온실가스 배출량을 감축하고 재생에너지 사용을 확대하며 이사회 산하 감사위원회의 독립성을 강화하였습니다"
test_nouns = extract_nouns(test_sentence, kiwi)
print(f"\n[ESG 문장 재검증]")
print(f"명사: {test_nouns}")

check_seeds = ["온실가스", "재생에너지", "이사회", "감사위원회", "독립성"]
print(f"\n[seed 보존 여부]")
for s in check_seeds:
    status = "✓" if s in test_nouns else "✗"
    print(f"  {s}: {status}")

# 더 까다로운 문장으로 추가 검증 (seed 단어가 조사/접미사와 붙은 경우)
hard_tests = [
    "재생에너지로 전환하고 폐기물을 재활용하며",
    "산업재해 예방과 중대재해 대응 체계를 마련",
    "사외이사의 독립성과 감사위원회의 전문성",
    "부패방지 및 준법 컴플라이언스 강화",
    "공급망 ESG 실사와 협력사 지원",
]
print(f"\n[까다로운 문장 검증]")
for sent in hard_tests:
    nouns = extract_nouns(sent, kiwi)
    print(f"\n  원문: {sent}")
    print(f"  명사: {nouns}")

# 전체 30개 seed가 '문장 속에서' 보존되는지 일괄 검증
print(f"\n[전체 30개 seed — 문장 삽입 검증]")
broken = []
for term in seed_terms:
    term = str(term).strip()
    # seed를 자연스러운 문맥에 넣어서 테스트
    test = f"당사는 {term} 관련 활동을 강화하였습니다"
    toks = extract_nouns(test, kiwi)
    if term not in toks:
        broken.append((term, toks))

if broken:
    print(f"  ⚠️ 문장 속 분리 {len(broken)}개:")
    for term, toks in broken:
        print(f"    {term} → {toks}")
else:
    print(f"  ✓ 30개 전부 문장 속에서도 보존")

사용자 사전 등록 (score=50): 30개

[ESG 문장 재검증]
명사: ['당사', '온실가스', '배출량', '감축', '재생에너지', '사용', '확대', '이사회', '산하', '감사위원회', '독립성', '강화']

[seed 보존 여부]
  온실가스: ✓
  재생에너지: ✓
  이사회: ✓
  감사위원회: ✓
  독립성: ✓

[까다로운 문장 검증]

  원문: 재생에너지로 전환하고 폐기물을 재활용하며
  명사: ['재생에너지', '전환', '폐기물', '재활용']

  원문: 산업재해 예방과 중대재해 대응 체계를 마련
  명사: ['산업재해', '예방과', '중대재해', '대응', '체계', '마련']

  원문: 사외이사의 독립성과 감사위원회의 전문성
  명사: ['사외이사', '독립성', '감사위원회', '전문']

  원문: 부패방지 및 준법 컴플라이언스 강화
  명사: ['부패방지', '준법', '컴플라이언스', '강화']

  원문: 공급망 ESG 실사와 협력사 지원
  명사: ['공급망', '실사', '협력사', '지원']

[전체 30개 seed — 문장 삽입 검증]
  ✓ 30개 전부 문장 속에서도 보존


### **2-4. 전체 corpus 토큰화 + 불용어 제거**

381개 firm-year 문서를 명사 토큰화한다 (seed 사전 등록된 Kiwi).
이어서 불용어를 제거한다.

**불용어 전략 (가이드 권장):**
- `data/stopwords_ko_esg.txt` 초안 활용 ("및", "관련", "통해" 등)
- 추가 제거: 회사명, 연도/숫자 표현, 반복 법률·회계 용어
- 불용어 제거는 분석 신호를 높이는 선택. 기준을 보고서에 기록

**처리량:** 381개 문서 (평균 3.4만 자). Kiwi가 빠르나 큰 문서
(POSCO 19만 자) 포함이라 수 분 소요 가능. 결과는 캐시.

In [34]:
# stopwords_ko_esg.txt 로드
stopwords_path = os.path.join(DATA_DIR, "stopwords_ko_esg.txt")
# 파일이 루트에 있을 수도 있으니 체크
if not os.path.exists(stopwords_path):
    stopwords_path = "stopwords_ko_esg.txt"

with open(stopwords_path, "r", encoding="utf-8") as f:
    base_stopwords = set(
        line.strip() for line in f 
        if line.strip() and not line.strip().startswith("#")
    )

print(f"기본 불용어 (stopwords_ko_esg.txt): {len(base_stopwords)}개")
print(f"예시 30개: {sorted(base_stopwords)[:30]}")

기본 불용어 (stopwords_ko_esg.txt): 33개
예시 30개: ['강화', '개선', '계획', '관련', '관리', '구축', '그리고', '기준', '당사', '대한', '등', '또한', '목표', '및', '보고서', '사업', '성과', '실시', '에게', '에서', '운영', '위해', '으로', '있습니다', '제공', '지원', '추진', '통해', '하는', '한다']


In [35]:
# 1) 회사명 불용어 — company_master의 모든 회사명에서 추출
#    "삼성전자", "삼성", "전자" 등이 ESG 신호 아닌데 자주 등장
company_names = cm["company_name"].unique().tolist()

# 회사명을 Kiwi로 토큰화해서 회사명 구성 명사도 불용어에 추가
company_stopwords = set()
for name in company_names:
    company_stopwords.add(str(name).strip())
    # 회사명의 명사 토큰도 (예: "삼성전자" → "삼성", "전자")
    for tok in extract_nouns(str(name), kiwi, min_len=2):
        company_stopwords.add(tok)

print(f"회사명 기반 불용어: {len(company_stopwords)}개")
print(f"예시: {sorted(company_stopwords)[:20]}")

# 2) 사업보고서 상투어 — ESG와 무관한 회계/법률 반복 용어
report_boilerplate = {
    "당사", "회사", "당기", "전기", "전년", "동기", "해당", "관련", "경우",
    "위함", "통해", "위해", "대한", "기준", "이상", "이하", "이내", "정도",
    "사항", "내용", "결과", "현황", "상황", "수준", "정보", "자료", "기타",
    "구분", "포함", "제외", "적용", "사용", "운영", "실시", "수행", "진행",
    "보고", "공시", "기재", "작성", "제출", "확인", "검토", "평가", "분석",
    "백만원", "천원", "억원", "원", "주", "년", "월", "일", "분기", "반기",
    "연결", "별도", "재무", "제표", "주석", "회계", "감사", "법인",
}

# 3) 사업보고서 공통 빈출 (코퍼스 분석으로 추후 보강 가능)
print(f"\n사업보고서 상투어: {len(report_boilerplate)}개")

# 전체 불용어 통합
all_stopwords = base_stopwords | company_stopwords | report_boilerplate
print(f"\n[통합 불용어] 총 {len(all_stopwords)}개")
print(f"  - 기본(파일): {len(base_stopwords)}")
print(f"  - 회사명 기반: {len(company_stopwords)}")
print(f"  - 상투어: {len(report_boilerplate)}")

# ⚠️ 중요: seed 단어가 불용어에 잘못 들어가지 않았는지 검증
seed_in_stop = [s for s in seed_terms if s in all_stopwords]
if seed_in_stop:
    print(f"\n⚠️ seed가 불용어에 포함됨 (제거 필요): {seed_in_stop}")
    all_stopwords -= set(seed_terms)
    print(f"  → seed 보호: 불용어에서 제외. 최종 {len(all_stopwords)}개")
else:
    print(f"\n✓ seed 단어는 불용어에 없음 (안전)")

회사명 기반 불용어: 182개
예시: ['BGF리테일', 'BNK금융지주', 'BYC', 'CJ', 'CJ ENM', 'CJ대한통운', 'CJ제일제당', 'CJ프레시웨이', 'CS홀딩스', 'DB하이텍', 'DL', 'GS글로벌', 'GS리테일', 'HD현대인프라코어', 'HL D&I', 'JW중외제약', 'KB금융', 'KB금융지주', 'KG케미칼', 'KR모터스']

사업보고서 상투어: 63개

[통합 불용어] 총 267개
  - 기본(파일): 33
  - 회사명 기반: 182
  - 상투어: 63

⚠️ seed가 불용어에 포함됨 (제거 필요): ['에너지']
  → seed 보호: 불용어에서 제외. 최종 266개


In [36]:
tokens_cache_path = os.path.join(OUTPUT_DIR, "corpus_tokens.json")

if os.path.exists(tokens_cache_path):
    print("[토큰 캐시 로드]")
    with open(tokens_cache_path, "r", encoding="utf-8") as f:
        token_data = json.load(f)
    print(f"  {len(token_data)}개 문서 토큰 로드")
else:
    print("[전체 corpus 토큰화 시작 — 수 분 소요]")
    token_data = {}
    t0 = time.time()
    
    for idx, row in corpus_df.iterrows():
        key = f"{row['stock_code']}_{row['fiscal_year']}"
        
        # 명사 추출
        nouns = extract_nouns(row["doc"], kiwi, min_len=2)
        # 불용어 제거
        filtered = [w for w in nouns if w not in all_stopwords]
        
        token_data[key] = filtered
        
        if (idx + 1) % 50 == 0:
            print(f"  {idx+1}/381 | 경과 {time.time()-t0:.0f}s")
    
    print(f"  완료. 총 {time.time()-t0:.0f}s")
    
    # 캐시 저장
    with open(tokens_cache_path, "w", encoding="utf-8") as f:
        json.dump(token_data, f, ensure_ascii=False)
    print(f"  저장: {tokens_cache_path}")

# 토큰화 결과 통계
token_counts = {k: len(v) for k, v in token_data.items()}
tc_series = pd.Series(token_counts)
print(f"\n[토큰 수 통계 (불용어 제거 후)]")
print(f"  평균: {tc_series.mean():,.0f}")
print(f"  중앙값: {tc_series.median():,.0f}")
print(f"  최소: {tc_series.min():,}")
print(f"  최대: {tc_series.max():,}")

# 샘플 토큰 확인
sample_key = list(token_data.keys())[0]
print(f"\n[샘플: {sample_key}]")
print(f"  토큰 수: {len(token_data[sample_key])}")
print(f"  앞 40개: {token_data[sample_key][:40]}")

# seed 단어가 실제 corpus 토큰에 등장하는지 확인
print(f"\n[seed 단어 corpus 등장 빈도 — 상위 확인]")
from collections import Counter
all_tokens_flat = [tok for toks in token_data.values() for tok in toks]
token_freq = Counter(all_tokens_flat)
for s in seed_terms[:15]:
    print(f"  {s}: {token_freq.get(s, 0):,}회")

[토큰 캐시 로드]
  381개 문서 토큰 로드

[토큰 수 통계 (불용어 제거 후)]
  평균: 5,265
  중앙값: 3,982
  최소: 570
  최대: 28,565

[샘플: 000020_2022]
  토큰 수: 2472
  앞 40개: ['일반', '지배', '기업', '실체', '재화', '용역', '근거', '영업', '부문', '부문', '내부', '목적', '활용', '제약', '의료', '기기', '제조', '판매업', '금융업', '영위', '영업', '부문', '금융업', '중요', '관점', '비중', '지배', '영업', '개황', '부문', '실체', '매출액', '대비', '증가', '영업', '이익', '대비', '증가', '재화', '용역']

[seed 단어 corpus 등장 빈도 — 상위 확인]
  탄소: 963회
  온실가스: 1,073회
  탄소중립: 423회
  넷제로: 9회
  재생에너지: 679회
  에너지: 2,851회
  전력: 1,026회
  폐기물: 894회
  재활용: 613회
  폐수: 329회
  안전: 1,866회
  산업재해: 48회
  중대재해: 74회
  임직원: 511회
  노동: 179회


### **2-5. Phase 2 마무리 — 토큰 품질 점검**

토큰화 완료. Phase 3 (TF-IDF) 진입 전 최종 점검:
- 전체 seed 30개 corpus 등장 빈도 (희소 seed 식별 → expanded 근거)
- 고빈도 일반명사 확인 (불용어 보강 필요성 판단)
- firm-year별 토큰 수와 원문 길이 상관 (정상성 확인)

In [37]:
# 전체 30개 seed 등장 빈도 (E/S/G별)
print("[전체 seed 30개 corpus 등장 빈도]\n")
for dim in ["E", "S", "G"]:
    terms = seed_df[seed_df["dimension"] == dim]["seed_term"].tolist()
    print(f"  [{dim} 차원]")
    for t in terms:
        freq = token_freq.get(t, 0)
        # 문서 등장 수 (몇 개 firm-year에 나오는지)
        doc_count = sum(1 for toks in token_data.values() if t in toks)
        flag = " ⚠️희소" if freq < 50 else ""
        print(f"    {t:>8}: {freq:>6,}회 | {doc_count:>3}/381 문서{flag}")
    print()

# 고빈도 일반명사 top 30 (불용어 보강 후보 — seed 제외)
print("[고빈도 토큰 Top 30 — seed 제외]")
seed_set = set(seed_terms)
non_seed_freq = {w: c for w, c in token_freq.items() if w not in seed_set}
top30 = sorted(non_seed_freq.items(), key=lambda x: -x[1])[:30]
for w, c in top30:
    print(f"  {w}: {c:,}")

[전체 seed 30개 corpus 등장 빈도]

  [E 차원]
          탄소:    963회 | 188/381 문서
        온실가스:  1,073회 | 152/381 문서
        탄소중립:    423회 |  96/381 문서
         넷제로:      9회 |   7/381 문서 ⚠️희소
       재생에너지:    679회 | 108/381 문서
         에너지:  2,851회 | 238/381 문서
          전력:  1,026회 | 155/381 문서
         폐기물:    894회 | 140/381 문서
         재활용:    613회 | 126/381 문서
          폐수:    329회 |  87/381 문서

  [S 차원]
          안전:  1,866회 | 292/381 문서
        산업재해:     48회 |  20/381 문서 ⚠️희소
        중대재해:     74회 |  27/381 문서
         임직원:    511회 | 190/381 문서
          노동:    179회 | 100/381 문서
          인권:     99회 |  35/381 문서
        교육훈련:     17회 |  17/381 문서 ⚠️희소
         협력사:    165회 |  53/381 문서
         공급망:    412회 | 141/381 문서
        지역사회:    146회 |  57/381 문서

  [G 차원]
         이사회:  7,928회 | 381/381 문서
        사외이사:  7,460회 | 381/381 문서
       감사위원회:  4,249회 | 362/381 문서
         독립성:  1,416회 | 372/381 문서
          윤리:    118회 |  81/381 문서
          준법:  1,676회 | 371/381 문서
      컴플라이언스:     

In [38]:
# firm-year별 토큰 수를 corpus_df에 병합
corpus_df["key"] = corpus_df["stock_code"] + "_" + corpus_df["fiscal_year"].astype(str)
corpus_df["n_tokens"] = corpus_df["key"].map(lambda k: len(token_data.get(k, [])))

# 원문 길이 vs 토큰 수 상관 (정상성 — 양의 상관이어야)
corr = corpus_df["doc_len"].corr(corpus_df["n_tokens"])
print(f"[원문 길이 vs 토큰 수 상관]: {corr:.4f}")
print(f"  (0.9+ 면 정상 — 긴 문서가 토큰도 많음)")

# total_word_count = n_tokens (Phase 3 cheap-talk 통제 변수로 사용)
print(f"\n[cheap-talk 통제 변수 후보: n_tokens]")
print(f"  평균 {corpus_df['n_tokens'].mean():,.0f}, "
      f"표준편차 {corpus_df['n_tokens'].std():,.0f}")
print(f"  최소 {corpus_df['n_tokens'].min():,}, 최대 {corpus_df['n_tokens'].max():,}")

# Phase 3 입력용 최종 데이터 구조 확인
print(f"\n[Phase 3 입력 데이터 준비 완료]")
print(f"  corpus_df: {corpus_df.shape}")
print(f"  컬럼: {list(corpus_df.columns)}")
print(f"  token_data: {len(token_data)}개 firm-year 토큰 리스트")
print(f"  seed_df: {seed_df.shape} (30개 seed, E/S/G)")

# 토큰을 공백 결합한 문서도 준비 (TfidfVectorizer 입력용)
corpus_df["tokens_joined"] = corpus_df["key"].map(
    lambda k: " ".join(token_data.get(k, []))
)
print(f"\n[TF-IDF 입력 형태 샘플]")
print(f"  {corpus_df.iloc[0]['key']}: {corpus_df.iloc[0]['tokens_joined'][:150]}...")

# 최종 점검: 메타와 corpus_df 일치
print(f"\n[최종 데이터 정합성]")
print(f"  corpus_df 행: {len(corpus_df)} (목표 381)")
print(f"  결측 토큰 문서: {(corpus_df['n_tokens']==0).sum()}개 (0이어야 정상)")
assert len(corpus_df) == 381
assert (corpus_df["n_tokens"] == 0).sum() == 0
print(f"  ✓ Phase 2 완료 — Phase 3 진입 준비 완료")

[원문 길이 vs 토큰 수 상관]: 0.9977
  (0.9+ 면 정상 — 긴 문서가 토큰도 많음)

[cheap-talk 통제 변수 후보: n_tokens]
  평균 5,265, 표준편차 4,783
  최소 570, 최대 28,565

[Phase 3 입력 데이터 준비 완료]
  corpus_df: (381, 8)
  컬럼: ['stock_code', 'fiscal_year', 'esg_year', 'rcept_no', 'doc', 'doc_len', 'key', 'n_tokens']
  token_data: 381개 firm-year 토큰 리스트
  seed_df: (30, 6) (30개 seed, E/S/G)

[TF-IDF 입력 형태 샘플]
  000020_2022: 일반 지배 기업 실체 재화 용역 근거 영업 부문 부문 내부 목적 활용 제약 의료 기기 제조 판매업 금융업 영위 영업 부문 금융업 중요 관점 비중 지배 영업 개황 부문 실체 매출액 대비 증가 영업 이익 대비 증가 재화 용역 근거 영업 부문 부문 내부 목적 활용 실체 제약...

[최종 데이터 정합성]
  corpus_df 행: 381 (목표 381)
  결측 토큰 문서: 0개 (0이어야 정상)
  ✓ Phase 2 완료 — Phase 3 진입 준비 완료


## 🔍 **핵심 발견 — ESG Seed Dictionary 분석 인사이트**

### **발견 1. 희소(seed) 단어 5개 — Expanded Dictionary 확장의 핵심 근거**

| Seed   | 빈도(회) | 등장 문서 수 | 해석         |
| ------ | ----: | ------: | ---------- |
| 넷제로    |     9 |       7 | 거의 사용되지 않음 |
| 산업재해   |    48 |      20 | 제한적으로 등장   |
| 교육훈련   |    17 |      17 | 가장 희소      |
| 부패방지   |     8 |       5 | 가장 희소      |
| 컴플라이언스 |    50 |      32 | 경계선 수준     |

→ 일부 seed 단어는 사업보고서 내 직접 등장 빈도가 매우 낮아, seed dictionary만으로는 ESG 신호를 충분히 포착하기 어려움.

이는 과제 가이드에서 요구한 **expanded dictionary 확장 필요성의 직접적 근거**

#### **대표 사례**

* **넷제로**
  → 국내 기업들은 ‘넷제로’보다 **탄소중립** 표현을 훨씬 더 많이 사용
  (예: 탄소중립 423회)

* **교육훈련**
  → 실제 문서에서는 안전교육, 직무교육, 역량개발 등 다양한 표현 사용

* **부패방지**
  → 반부패, 청렴, 윤리경영 등의 표현으로 대체되는 경우 다수

#### **핵심 해석**

→ FastText 기반 확장을 통해 희소 seed의 유사어를 추가함으로써 ESG 표현 다양성을 보완할 수 있음.
→ 특히 희소 단어의 semantic expansion이 expanded dictionary 구축의 핵심 역할을 수행함.

---

### **발견 2. G(Governance) 차원 seed의 압도적 빈도**

| 단어    | 빈도(회) | 등장 문서 수 |
| ----- | ----: | ------: |
| 이사회   | 7,928 |     381 |
| 사외이사  | 7,460 |     381 |
| 주주    | 7,329 |     381 |
| 감사위원회 | 4,249 |     362 |

→ G(지배구조) 관련 단어는 거의 모든 사업보고서에서 반복적으로 등장함.

이는 이사회 구성, 사외이사 현황, 감사위원회 등이 법정 공시 항목이기 때문으로 해석된다.

#### **핵심 해석**

→ G seed 점수가 높다고 해서 반드시 실제 지배구조 수준이 우수하다고 해석할 수 없음.

즉, G 차원은:

* 기업 자발적 ESG 활동보다
* 법적·제도적 공시 의무의 영향을 크게 받는 영역임.

따라서 G 점수 해석 시에는 **cheap-talk 위험**에 대한 주의가 필요하다.

---

### **발견 3. 차원(E/S/G) 간 극단적 빈도 불균형**

예시:

* 이사회: 7,928회
* 교육훈련: 17회

→ 약 **466배 차이**

#### **핵심 해석**

ESG 차원 간 단어 빈도 자체가 크게 다르므로

* G 합계 ≫ E 합계 > S 합계 구조가 나타남
* 단순 raw frequency 비교는 왜곡 가능성이 큼

TF-IDF가 일부 보정을 수행하더라도,

* E/S/G 점수를 서로 직접 비교하는 것은 적절하지 않음
* 각 차원 내부에서 기업 간 상대 비교 방식이 더 타당함

따라서:

* 회귀분석 및 score 해석도 차원별(E/S/G separately) 접근이 필요함.

---

### **발견 4. 고빈도 일반 경영 용어(Noise) 존재**

| 단어 |     빈도 |
| -- | -----: |
| 시장 | 25,138 |
| 위험 | 22,651 |
| 자산 | 18,471 |
| 영업 | 13,940 |

→ 사업보고서에는 ESG와 직접 관련 없는 일반 재무·경영 용어가 매우 높은 빈도로 등장함.

#### **핵심 해석**

* Seed TF-IDF 점수에는 직접 영향이 크지 않음
  (seed 단어만 합산하기 때문)

하지만:

* 기준 문장(sentence embedding)
* cosine similarity 기반 분석

에서는 이러한 일반 고빈도 단어가 노이즈로 작용할 가능성이 존재함.

* stopword 정제
* TF-IDF weighting
* domain-specific filtering

의 중요성이 확인된다.


---
## **Phase 1 — 데이터 수집 (381/381 성공)**

### **1-1. 핵심 설계 결정과 근거**

- `company_master.csv`에서 `company_name`은 133개인데 `stock_code`는 127개다.
  사명 변경·분할·존속회사 표기 차이 때문에 같은 종목코드가 여러 이름으로
  나타난다. 회사명을 join 키로 쓰면 조용한 오매칭(silent mismatch)이 생긴다.
- `corp_code`, `rcept_no`는 CSV에 전부 비어 있어(381/381 결측) 직접
  채워야 한다.

수집 도구로 OpenDART API 직접 호출을 선택했다.

- 선택지는 MCP / CLI(dart-fss) / API 직접이었다. API 직접 호출이 재현성이
  가장 높고, MCP passage extraction은 LLM이 생성한 텍스트라 원문과 다를 수
  있다는 가이드 경고가 결정적이었다.
- `corpCode.xml` 전체(약 10만 기업)를 1회 다운로드해 `stock_code → corp_code`
  매핑을 일괄 구축했다 (예: 삼성전자 corp_code = 00126380).

### **1-2. 타이밍 규칙 처리**

가이드의 `esg_year = fiscal_year + 1` 규칙과 실제 CSV가 달랐다.

- 가이드는 "2025 ESG 등급 ↔ 2024.12 사업보고서"라고 명시하지만, CSV의
  `esg_year`는 `fiscal_year`와 같은 값(둘 다 2022~2024)으로 표기돼 있었다.
- 이를 사업보고서 검색 시점 처리로 해석했다. `fiscal_year=2024`
  보고서는 2025년에 공시되므로 DART 검색 범위를 `fiscal_year + 1`년으로
  잡았다(`bgn_de=20250101`, `end_de=20251231`).

### **1-3. 수집 함수 3종**

| 함수 | 역할 |
| --- | --- |
| `find_business_report_v2()` | corp_code로 fiscal_year+1년 공시 사업보고서 검색, rcept_no 반환 |
| `download_document_xml()` | rcept_no로 원문 ZIP 다운로드 → 본 보고서 XML 추출 |
| `extract_esg_sections()` | XML에서 II/IV/VI 섹션의 `<P>` 텍스트만 추출 |

`collect_one_firm_year_v2()`가 위 3개를 묶어 1개 firm-year를 완전 수집한다.

### **1-4. 실패 13건 → 전수 복구**

초기 수집은 368/381 성공, 13건 실패였다. 실패를 가짜 0으로 채우지 않고
원인을 직접 진단한 결과 두 가지 버그를 발견했다.

- **결산월 하드코딩 문제**: 초기 코드는 보고서명에 `(YYYY.12)`가 포함된
  것만 찾았다. 신영증권은 3월 결산, 만호제강은 6월 결산이라
  `(2025.03)`, `(2025.06)`로 공시돼 검색에서 누락됐다.
  → 결산월 무관하게 "사업보고서" 포함이면 매칭하도록 수정.
- **정정공시 문제**: `[첨부정정]` 보고서는 document.xml에 원문이 없을 수
  있다(status 014). → 정정 우선순위를 **원본 > [기재정정] > [첨부정정]**로
  정렬해 원문이 확실한 버전을 우선 선택.

수정 함수로 13건 전부 원본 보고서 재매칭 → 다운로드 → 381개 전수 성공

### **1-5. 산출물**

- `collection_meta.csv` (381행): 수집 lineage 전체 기록 (rcept_no, 정정
  우선순위, 섹션별 글자 수, viewer URL)
- `corpus/*.json` (381개): firm-year별 II/IV/VI 통합 텍스트

## **Phase 2 — 한국어 전처리**

### **2-1. 형태소 분석기: Kiwi, 명사만 추출**

- Kiwi(kiwipiepy)를 선택했다. C++ 기반으로 빠르고 최신 사전을 반영한다.
- seed 30개가 전부 명사이고 TF-IDF가 목적이므로 명사(NNG/NNP)만 추출
  (`extract_nouns()`). 1글자 명사·숫자 단독은 노이즈라 제외(min_len=2).

### **2-2. 복합명사 보존 — seed 사용자 사전 등록 (핵심 분석 선택)**

Kiwi 기본 분석은 복합명사를 분리한다. `재생에너지` → `재생` + `에너지`,
`감사위원회` → `감사` + `위원회`로 깨지면 seed 단어를 측정할 수 없다.

- 해결: 30개 seed를 `kiwi.add_user_word(term, "NNP", score=50.0)`로 등록.
  score를 높게 줘 문맥 속에서도 강제 우선 인식되게 했다.
- 까다로운 문장(조사·접미사가 붙은 경우)까지 검증해 **30개 seed 전부
  문장 속에서 단일 토큰으로 보존**되는 것을 확인했다.

### **2-3. 불용어 — 266개, seed 보호 로직 필수**

세 종류를 통합했다.

- 기본 불용어 파일(`stopwords_ko_esg.txt`)
- 회사명 기반: company_master 전 회사명을 토큰화해 추출 (예: "삼성전자"
  → "삼성", "전자")
- 사업보고서 상투어: "당사", "회사", "백만원" 등 ESG 무관 반복 용어

**중요 — seed 보호**: 회사명에 "에너지"가 들어간 기업이 있어 seed 단어
`에너지`가 회사명 기반 불용어에 섞여 들어갔다. 이를 자동 감지해 seed는
불용어에서 무조건 제외하는 안전장치를 넣었다. 이 검증이 없었으면 E 차원
신호가 손상됐을 것이다.

### **2-4. 토큰화 결과 정합성**

- 381개 문서 토큰화 완료. `corpus_tokens.json`으로 캐시
- 원문 길이 vs 토큰 수 상관 ≈ 0.998 (정상 — 긴 문서가 토큰도 많음)
- 토큰 수가 **firm-year마다 크게 다름**(최소 수천 ~ 최대 수십만, 약 45배
  편차). 이 변동은 Phase 3에서 cheap-talk 통제 변수(n_tokens)로 쓴다.

## **Phase 2 분석 인사이트**

| # | 발견 | 후속 분석에 주는 함의 |
| --- | --- | --- |
| 1 | 희소 seed 5개 (넷제로 9회, 부패방지 8회, 교육훈련 17회 등) | seed-only로는 신호 약함 → **expanded dictionary(FastText 확장)의 직접 근거** |
| 2 | G seed 압도적 빈도 (이사회 7,928회, 사외이사 7,460회 — 381/381 전 문서 등장) | 법정 의무 공시 → G 점수 높다고 지배구조 우수 아님. **cheap-talk 위험** |
| 3 | 차원 간 빈도 466배 불균형 (이사회 7,928 vs 교육훈련 17) | E/S/G 점수 직접 비교 금지. **차원 내 상대 비교 + 차원별 별도 회귀** |
| 4 | 고빈도 일반 경영 용어 존재 (시장 25,138, 위험 22,651) | seed 합산엔 영향 작으나 cosine similarity 분석엔 노이즈. 추가 정제 필요 |

### **다음 단계 할 일**

- `corpus_tokens.json` (381개 firm-year 토큰 리스트) — TF-IDF 입력
- `seed_df` (30개 seed, E/S/G) — 점수 합산 기준
- `n_tokens` — cheap-talk 통제 변수
- 발견 1~4 — feature 설계와 해석 시 그대로 반영해야 할 사전 진단